In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, to_date, when, lit, coalesce, size, split, explode, collect_set, array_contains, array, regexp_replace, round, regexp_extract
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
import re
import yaml

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/05 22:25:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## EDA

In [4]:
# Path to your raw data folder
data_folder = "data"

# Loop through all CSV files in the folder
for file in os.listdir(data_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(data_folder, file)
        try:
            # Load just the header (fast, without reading the full file)
            df = pd.read_csv(file_path, nrows=0)
            print(f"\n📄 File: {file}")
            print("Columns:", list(df.columns))
        except Exception as e:
            print(f"Error reading {file}: {e}")



📄 File: features_attributes.csv
Columns: ['Customer_ID', 'Name', 'Age', 'SSN', 'Occupation', 'snapshot_date']

📄 File: features_financials.csv
Columns: ['Customer_ID', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance', 'snapshot_date']

📄 File: feature_clickstream.csv
Columns: ['fe_1', 'fe_2', 'fe_3', 'fe_4', 'fe_5', 'fe_6', 'fe_7', 'fe_8', 'fe_9', 'fe_10', 'fe_11', 'fe_12', 'fe_13', 'fe_14', 'fe_15', 'fe_16', 'fe_17', 'fe_18', 'fe_19', 'fe_20', 'Customer_ID', 'snapshot_date']

📄 File: lms_loan_daily.csv
Columns: ['loan_id', 'Customer_ID', 'loan_start_date', 'tenure', 'installment_num', 'loan_amt', 'due_amt', 'paid_a

In [5]:
clickstream_df = pd.read_csv("data/feature_clickstream.csv")

In [23]:
sorted(clickstream_df['snapshot_date'].unique())

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

In [11]:
clickstream_df.describe(include='all')

,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
count,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,...,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376,215376
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8974,24
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CUS_0xfe4,2023-01-01
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,8974
mean,101.414796,103.096195,104.333709,105.648503,106.996676,103.235922,107.070337,110.718724,114.406354,117.775797,...,100.420804,100.025801,99.600383,99.860834,100.129425,100.125139,100.341162,100.046259,NaN,NaN
std,99.833594,99.930002,100.599865,100.326065,100.693607,100.270388,100.323265,100.243698,100.186139,100.807686,...,100.881826,101.070371,101.145906,100.751876,101.298428,102.231587,102.666804,103.589358,NaN,NaN
min,-378.000000,-356.000000,-399.000000,-307.000000,-343.000000,-321.000000,-368.000000,-361.000000,-328.000000,-317.000000,...,-355.000000,-394.000000,-351.000000,-342.000000,-329.000000,-344.000000,-401.000000,-354.000000,NaN,NaN
25%,34.000000,36.000000,36.000000,38.000000,39.000000,36.000000,39.000000,43.000000,47.000000,50.000000,...,33.000000,32.000000,31.000000,32.000000,32.000000,31.000000,31.000000,30.000000,NaN,NaN
50%,102.000000,103.000000,104.000000,106.000000,107.000000,103.000000,107.000000,111.000000,115.000000,118.000000,...,101.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,NaN,NaN
75%,169.000000,171.000000,172.000000,173.000000,175.000000,171.000000,174.000000,179.000000,182.000000,186.000000,...,168.000000,168.000000,168.000000,168.000000,169.000000,169.000000,170.000000,170.000000,NaN,NaN


In [6]:
clickstream_df

,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
0,63,118,80,121,55,193,111,112,-101,83,...,-16,-81,-126,114,35,85,-73,76,CUS_0x1037,2023-01-01
1,-108,182,123,4,-56,27,25,-6,284,222,...,-14,-96,200,35,130,94,111,75,CUS_0x1069,2023-01-01
2,-13,8,87,166,214,-98,215,152,129,139,...,26,86,171,125,-130,354,17,302,CUS_0x114a,2023-01-01
3,-85,45,200,89,128,54,76,51,61,139,...,172,96,174,163,37,207,180,118,CUS_0x1184,2023-01-01
4,55,120,226,-86,253,97,107,68,103,126,...,76,43,183,159,-26,104,118,184,CUS_0x1297,2023-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215371,414,22,72,57,142,192,11,139,24,63,...,179,91,20,189,-35,-19,15,66,CUS_0xdf6,2024-12-01
215372,116,-124,-108,212,-21,227,146,112,186,-65,...,38,226,319,98,9,152,17,14,CUS_0xe23,2024-12-01
215373,237,-3,-49,375,144,41,-170,324,19,266,...,7,102,64,191,124,220,231,75,CUS_0xe4e,2024-12-01
215374,5,67,211,83,207,-41,325,14,-18,41,...,109,266,28,157,131,116,101,131,CUS_0xedd,2024-12-01


In [28]:
clickstream_df['snapshot_date'].value_counts(dropna=False)

snapshot_date
2023-01-01    8974
2023-02-01    8974
2023-03-01    8974
2023-04-01    8974
2023-05-01    8974
2023-06-01    8974
2023-07-01    8974
2023-08-01    8974
2023-09-01    8974
2023-10-01    8974
2023-11-01    8974
2023-12-01    8974
2024-01-01    8974
2024-02-01    8974
2024-03-01    8974
2024-04-01    8974
2024-05-01    8974
2024-06-01    8974
2024-07-01    8974
2024-08-01    8974
2024-09-01    8974
2024-10-01    8974
2024-11-01    8974
2024-12-01    8974
Name: count, dtype: int64

In [31]:
clickstream_df['Customer_ID'].isna().any()

np.False_

In [7]:
attributes_df = pd.read_csv("data/features_attributes.csv")

In [22]:
sorted(attributes_df['snapshot_date'].unique())

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01',
 '2025-01-01']

In [12]:
attributes_df.describe(include='all')

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
count,12500,12500,12500,12500,12500,12500
unique,12500,10139,301,11798,16,25
top,CUS_0xffd,Langep,26,#F%$D@*&8,_______,2024-08-01
freq,1,6,364,703,880,543


In [17]:
attributes_df['snapshot_date'].value_counts(dropna=False)

snapshot_date
2024-08-01    543
2023-01-01    530
2025-01-01    526
2023-05-01    521
2024-02-01    518
2023-06-01    517
2024-12-01    515
2024-04-01    513
2024-03-01    511
2023-04-01    510
2023-03-01    506
2024-07-01    505
2023-02-01    501
2024-06-01    498
2024-09-01    493
2023-11-01    491
2024-05-01    491
2023-12-01    489
2024-11-01    488
2023-10-01    487
2024-01-01    485
2023-08-01    481
2023-07-01    471
2024-10-01    456
2023-09-01    454
Name: count, dtype: int64

In [8]:
attributes_df['Customer_ID'].isna().any()

np.False_

In [9]:
financials_df = pd.read_csv("data/features_financials.csv")

In [21]:
sorted(financials_df['snapshot_date'].unique())

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01',
 '2025-01-01']

In [36]:
financials_df[financials_df['snapshot_date']=='2025-01-01']

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
1,CUS_0x1009,52312.68_,4250.390000,6,5,17,4,"Not Specified, Home Equity Loan, Credit-Builde...",5,18,...,_,202.68,40.286997,31 Years and 0 Months,Yes,108.366467,58.66019164829086,High_spent_Medium_value_payments,508.01234122645366,2025-01-01
34,CUS_0x1098,20652.98_,1468.081667,7,5,21,2,"Auto Loan, and Payday Loan",24,10,...,Standard,1859.53,37.441101,18 Years and 11 Months,NM,33.238356,54.60338563277085,Low_spent_Small_value_payments,348.96642479393626,2025-01-01
37,CUS_0x109f,130435.86000000002,10623.655000,4,846,9,2,"Home Equity Loan, and Home Equity Loan",13,1,...,Good,942.71,34.784704,29 Years and 11 Months,NM,188.186133,1070.730679227028,Low_spent_Medium_value_payments,81.1419359951725,2025-01-01
42,CUS_0x10b6,5691341.0,1712.969167,7,3,19,3,"Payday Loan, Credit-Builder Loan, and Debt Con...",6,12,...,Standard,277.93,40.278541,18 Years and 4 Months,No,37.996378,65.372164147804,High_spent_Small_value_payments,327.9283743913377,2025-01-01
49,CUS_0x10e2,43133.85000000001,3432.487500,10,10,31,7,"Personal Loan, Auto Loan, Not Specified, Home ...",43,24,...,Bad,2504.1,27.115861,10 Years and 9 Months,Yes,151.287255,67.89250966068741,High_spent_Large_value_payments,364.0689853638198,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12439,CUS_0xf15,45652.17,3888.347500,8,6,13,7,"Mortgage Loan, Payday Loan, Debt Consolidation...",10,10,...,_,28.54,26.846632,10 Years and 11 Months,Yes,160.473304,154.47086437898875,Low_spent_Medium_value_payments,353.8905820472088,2025-01-01
12459,CUS_0xf60,34413.76,2916.813333,6,5,8,5,"Home Equity Loan, Home Equity Loan, Home Equit...",23,20,...,_,113.06,24.508003,6 Years and 4 Months,Yes,74.328709,330.7631176311532,Low_spent_Small_value_payments,176.5895062534961,2025-01-01
12463,CUS_0xf74,17111.35,1367.645556,10,6,3868,7,"Personal Loan, Mortgage Loan, Home Equity Loan...",55,25,...,Bad,1763.48,25.724679,19 Years and 0 Months,Yes,95.734001,55.60022588346901,Low_spent_Medium_value_payments,264.66063434363843,2025-01-01
12471,CUS_0xf95,15604.56,1389.380000,9,5,25,3,"Debt Consolidation Loan, Home Equity Loan, and...",27,23,...,Bad,2686.18,35.184082,19 Years and 10 Months,Yes,37.912350,92.1139885286824,Low_spent_Small_value_payments,298.9116613286117,2025-01-01


In [13]:
financials_df.describe(include='all')

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
count,12500,12500,12500.000000,12500.000000,12500.000000,12500.000000,12500,11074,12500.000000,12500,...,12500,12500,12500.000000,12500,12500,12500.000000,12500,12500,12500,12500
unique,12500,12492,NaN,NaN,NaN,NaN,87,6260,NaN,168,...,4,12213,NaN,393,3,NaN,11921,7,12500,25
top,CUS_0xffd,17273.83,NaN,NaN,NaN,NaN,3,Not Specified,NaN,19,...,Standard,1360.45,NaN,16 Years and 2 Months,Yes,NaN,__10000__,Low_spent_Small_value_payments,389.434630761722,2024-08-01
freq,1,2,NaN,NaN,NaN,NaN,1808,176,NaN,735,...,4497,3,NaN,79,6571,NaN,558,3202,1,543
mean,NaN,NaN,4188.592303,16.939920,23.172720,73.213360,NaN,NaN,21.060880,NaN,...,NaN,NaN,32.349265,NaN,NaN,1488.394291,NaN,NaN,NaN,NaN
std,NaN,NaN,3180.147611,114.350815,132.005866,468.682227,NaN,NaN,14.863091,NaN,...,NaN,NaN,5.156815,NaN,NaN,8561.449910,NaN,NaN,NaN,NaN
min,NaN,NaN,303.645417,-1.000000,0.000000,1.000000,NaN,NaN,-5.000000,NaN,...,NaN,NaN,20.100770,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,1624.937917,3.000000,4.000000,8.000000,NaN,NaN,10.000000,NaN,...,NaN,NaN,28.066517,NaN,NaN,31.496968,NaN,NaN,NaN,NaN
50%,NaN,NaN,3087.595000,6.000000,5.000000,14.000000,NaN,NaN,18.000000,NaN,...,NaN,NaN,32.418953,NaN,NaN,72.887628,NaN,NaN,NaN,NaN
75%,NaN,NaN,5947.364167,7.000000,7.000000,20.000000,NaN,NaN,28.000000,NaN,...,NaN,NaN,36.623650,NaN,NaN,169.634826,NaN,NaN,NaN,NaN


In [18]:
financials_df['snapshot_date'].value_counts(dropna=False)

snapshot_date
2024-08-01    543
2023-01-01    530
2025-01-01    526
2023-05-01    521
2024-02-01    518
2023-06-01    517
2024-12-01    515
2024-04-01    513
2024-03-01    511
2023-04-01    510
2023-03-01    506
2024-07-01    505
2023-02-01    501
2024-06-01    498
2024-09-01    493
2023-11-01    491
2024-05-01    491
2023-12-01    489
2024-11-01    488
2023-10-01    487
2024-01-01    485
2023-08-01    481
2023-07-01    471
2024-10-01    456
2023-09-01    454
Name: count, dtype: int64

In [33]:
financials_df['Customer_ID'].isna().any()

np.False_

In [10]:
lms_loan_daily_df = pd.read_csv("data/lms_loan_daily.csv")

In [14]:
lms_loan_daily_df.describe(include='all')

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
count,137500,137500,137500,137500.0,137500.000000,137500.0,137500.000000,137500.000000,137500.000000,137500.000000,137500
unique,12500,12500,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35
top,CUS_0xffd_2024_03_01,CUS_0xffd,2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01
freq,11,11,5973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5539
mean,NaN,NaN,NaN,10.0,5.000000,10000.0,909.090909,711.810909,871.905455,5871.905455,NaN
std,NaN,NaN,NaN,0.0,3.162289,0.0,287.480833,485.726859,2002.672544,3070.777575,NaN
min,NaN,NaN,NaN,10.0,0.000000,10000.0,0.000000,0.000000,0.000000,0.000000,NaN
25%,NaN,NaN,NaN,10.0,2.000000,10000.0,1000.000000,0.000000,0.000000,3000.000000,NaN
50%,NaN,NaN,NaN,10.0,5.000000,10000.0,1000.000000,1000.000000,0.000000,7000.000000,NaN
75%,NaN,NaN,NaN,10.0,8.000000,10000.0,1000.000000,1000.000000,0.000000,8000.000000,NaN


In [20]:
sorted(lms_loan_daily_df['snapshot_date'].unique())

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01',
 '2025-01-01',
 '2025-02-01',
 '2025-03-01',
 '2025-04-01',
 '2025-05-01',
 '2025-06-01',
 '2025-07-01',
 '2025-08-01',
 '2025-09-01',
 '2025-10-01',
 '2025-11-01']

In [24]:
sorted(lms_loan_daily_df['loan_start_date'].unique())

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01',
 '2025-01-01']

In [35]:
lms_loan_daily_df[lms_loan_daily_df['loan_start_date']=='2025-01-01']

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
11,CUS_0x1009_2025_01_01,CUS_0x1009,2025-01-01,10,0,10000,0.0,0.0,0.0,10000.0,2025-01-01
12,CUS_0x1009_2025_01_01,CUS_0x1009,2025-01-01,10,1,10000,1000.0,1000.0,0.0,9000.0,2025-02-01
13,CUS_0x1009_2025_01_01,CUS_0x1009,2025-01-01,10,2,10000,1000.0,1000.0,0.0,8000.0,2025-03-01
14,CUS_0x1009_2025_01_01,CUS_0x1009,2025-01-01,10,3,10000,1000.0,1000.0,0.0,7000.0,2025-04-01
15,CUS_0x1009_2025_01_01,CUS_0x1009,2025-01-01,10,4,10000,1000.0,1000.0,0.0,6000.0,2025-05-01
...,...,...,...,...,...,...,...,...,...,...,...
137396,CUS_0xfdf_2025_01_01,CUS_0xfdf,2025-01-01,10,6,10000,1000.0,1000.0,0.0,4000.0,2025-07-01
137397,CUS_0xfdf_2025_01_01,CUS_0xfdf,2025-01-01,10,7,10000,1000.0,1000.0,0.0,3000.0,2025-08-01
137398,CUS_0xfdf_2025_01_01,CUS_0xfdf,2025-01-01,10,8,10000,1000.0,1000.0,0.0,2000.0,2025-09-01
137399,CUS_0xfdf_2025_01_01,CUS_0xfdf,2025-01-01,10,9,10000,1000.0,1000.0,0.0,1000.0,2025-10-01


In [17]:
lms_loan_daily_df.head(50)

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
0,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,0,10000,0.0,0.0,0.0,10000.0,2023-05-01
1,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,1,10000,1000.0,1000.0,0.0,9000.0,2023-06-01
2,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,2,10000,1000.0,1000.0,0.0,8000.0,2023-07-01
3,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,3,10000,1000.0,0.0,1000.0,8000.0,2023-08-01
4,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,4,10000,1000.0,2000.0,0.0,6000.0,2023-09-01
5,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,5,10000,1000.0,0.0,1000.0,6000.0,2023-10-01
6,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,6,10000,1000.0,0.0,2000.0,6000.0,2023-11-01
7,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,7,10000,1000.0,0.0,3000.0,6000.0,2023-12-01
8,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,8,10000,1000.0,0.0,4000.0,6000.0,2024-01-01
9,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,9,10000,1000.0,0.0,5000.0,6000.0,2024-02-01


In [19]:
lms_loan_daily_df['snapshot_date'].value_counts(dropna=False)

snapshot_date
2025-01-01    5539
2024-09-01    5537
2024-08-01    5531
2024-12-01    5531
2024-10-01    5502
2024-11-01    5501
2023-11-01    5469
2024-07-01    5442
2023-12-01    5428
2024-03-01    5425
2024-02-01    5424
2024-06-01    5418
2024-04-01    5417
2024-01-01    5412
2024-05-01    5391
2025-02-01    5028
2023-10-01    4978
2025-03-01    4515
2023-09-01    4491
2023-08-01    4037
2025-04-01    4024
2023-07-01    3556
2025-05-01    3526
2023-06-01    3085
2025-06-01    3021
2023-05-01    2568
2025-07-01    2478
2023-04-01    2047
2025-08-01    1985
2023-03-01    1537
2025-09-01    1529
2025-10-01    1041
2023-02-01    1031
2023-01-01     530
2025-11-01     526
Name: count, dtype: int64

In [39]:
lms_loan_daily_df['Customer_ID'].isna().any()

np.False_

In [49]:
lms_loan_daily_df['Customer_ID_start_date'] = lms_loan_daily_df['Customer_ID']+lms_loan_daily_df['loan_start_date']

In [38]:
financials_df['Customer_ID_Snapshot'] = financials_df['Customer_ID']+financials_df['snapshot_date']

In [40]:
attributes_df['Customer_ID_Snapshot'] = attributes_df['Customer_ID']+attributes_df['snapshot_date']

In [73]:
set1 = set(financials_df['Customer_ID_Snapshot'])
set2 = set(attributes_df['Customer_ID_Snapshot'])
set3 = set(lms_loan_daily_df['Customer_ID_start_date'])
set1 == set2 == set3

True

In [66]:
cust_set = set(financials_df['Customer_ID'])

In [67]:
len(cust_set)

12500

In [57]:
ft_2025_set = set(financials_df[financials_df['snapshot_date']=='2025-01-01']['Customer_ID'])

In [58]:
len(ft_2025_set)

526

In [59]:
ft_ex_2025_set = set(financials_df['Customer_ID']) - ft_2025_set

In [60]:
len(ft_ex_2025_set)

11974

In [61]:
click_cust_set = set(clickstream_df['Customer_ID'])

In [62]:
len(click_cust_set)

8974

In [72]:
common_set = (click_cust_set & ft_ex_2025_set)
len(common_set)

8974

In [68]:
len(cust_set-click_cust_set)

3526

In [69]:
clickstream_df['Customer_ID_Snapshot'] = clickstream_df['Customer_ID'] + clickstream_df['snapshot_date']

In [71]:
set4 = clickstream_df['Customer_ID_Snapshot']
len(set4)

215376

In [75]:
lms_loan_daily_df['loan_start_date'].value_counts()

loan_start_date
2024-08-01    5973
2023-01-01    5830
2025-01-01    5786
2023-05-01    5731
2024-02-01    5698
2023-06-01    5687
2024-12-01    5665
2024-04-01    5643
2024-03-01    5621
2023-04-01    5610
2023-03-01    5566
2024-07-01    5555
2023-02-01    5511
2024-06-01    5478
2024-09-01    5423
2023-11-01    5401
2024-05-01    5401
2023-12-01    5379
2024-11-01    5368
2023-10-01    5357
2024-01-01    5335
2023-08-01    5291
2023-07-01    5181
2024-10-01    5016
2023-09-01    4994
Name: count, dtype: int64

In [80]:
lms_loan_daily_df[lms_loan_daily_df['loan_start_date']
<= '2023-03-01']['Customer_ID'].nunique()

1537

## Main

In [6]:
# Define input/output directories
raw_data_dir = "/app/data"
bronze_dir = "/app/datamart/bronze"
silver_dir = "/app/datamart/silver"
gold_dir = "/app/datamart/gold"

In [7]:
# Run the processing function
process_bronze_tables(raw_data_dir, bronze_dir, spark)

Processing features_attributes.csv...
📊 Row counts per snapshot_date for features_attributes:
   2023-01-01: 530 rows
   2023-02-01: 501 rows
   2023-03-01: 506 rows
   2023-04-01: 510 rows
   2023-05-01: 521 rows
   2023-06-01: 517 rows
   2023-07-01: 471 rows
   2023-08-01: 481 rows
   2023-09-01: 454 rows
   2023-10-01: 487 rows
   2023-11-01: 491 rows
   2023-12-01: 489 rows
   2024-01-01: 485 rows
   2024-02-01: 518 rows
   2024-03-01: 511 rows
   2024-04-01: 513 rows
   2024-05-01: 491 rows
   2024-06-01: 498 rows
   2024-07-01: 505 rows
   2024-08-01: 543 rows
   2024-09-01: 493 rows
   2024-10-01: 456 rows
   2024-11-01: 488 rows
   2024-12-01: 515 rows
   2025-01-01: 526 rows


✅ Saved Bronze table: /app/datamart/bronze/features_attributes
Processing features_financials.csv...
📊 Row counts per snapshot_date for features_financials:
   2023-01-01: 530 rows
   2023-02-01: 501 rows
   2023-03-01: 506 rows
   2023-04-01: 510 rows
   2023-05-01: 521 rows
   2023-06-01: 517 rows
   2023-07-01: 471 rows
   2023-08-01: 481 rows
   2023-09-01: 454 rows
   2023-10-01: 487 rows
   2023-11-01: 491 rows
   2023-12-01: 489 rows
   2024-01-01: 485 rows
   2024-02-01: 518 rows
   2024-03-01: 511 rows
   2024-04-01: 513 rows
   2024-05-01: 491 rows
   2024-06-01: 498 rows
   2024-07-01: 505 rows
   2024-08-01: 543 rows
   2024-09-01: 493 rows
   2024-10-01: 456 rows
   2024-11-01: 488 rows
   2024-12-01: 515 rows
   2025-01-01: 526 rows


✅ Saved Bronze table: /app/datamart/bronze/features_financials
Processing feature_clickstream.csv...
📊 Row counts per snapshot_date for feature_clickstream:
   2023-01-01: 8974 rows
   2023-02-01: 8974 rows
   2023-03-01: 8974 rows
   2023-04-01: 8974 rows
   2023-05-01: 8974 rows
   2023-06-01: 8974 rows
   2023-07-01: 8974 rows
   2023-08-01: 8974 rows
   2023-09-01: 8974 rows
   2023-10-01: 8974 rows
   2023-11-01: 8974 rows
   2023-12-01: 8974 rows
   2024-01-01: 8974 rows
   2024-02-01: 8974 rows
   2024-03-01: 8974 rows
   2024-04-01: 8974 rows
   2024-05-01: 8974 rows
   2024-06-01: 8974 rows
   2024-07-01: 8974 rows
   2024-08-01: 8974 rows
   2024-09-01: 8974 rows
   2024-10-01: 8974 rows
   2024-11-01: 8974 rows
   2024-12-01: 8974 rows


✅ Saved Bronze table: /app/datamart/bronze/feature_clickstream
Processing lms_loan_daily.csv...
📊 Row counts per snapshot_date for lms_loan_daily:
   2023-01-01: 530 rows
   2023-02-01: 1031 rows
   2023-03-01: 1537 rows
   2023-04-01: 2047 rows
   2023-05-01: 2568 rows
   2023-06-01: 3085 rows
   2023-07-01: 3556 rows
   2023-08-01: 4037 rows
   2023-09-01: 4491 rows
   2023-10-01: 4978 rows
   2023-11-01: 5469 rows
   2023-12-01: 5428 rows
   2024-01-01: 5412 rows
   2024-02-01: 5424 rows
   2024-03-01: 5425 rows
   2024-04-01: 5417 rows
   2024-05-01: 5391 rows
   2024-06-01: 5418 rows
   2024-07-01: 5442 rows
   2024-08-01: 5531 rows
   2024-09-01: 5537 rows
   2024-10-01: 5502 rows
   2024-11-01: 5501 rows
   2024-12-01: 5531 rows
   2025-01-01: 5539 rows
   2025-02-01: 5028 rows
   2025-03-01: 4515 rows
   2025-04-01: 4024 rows
   2025-05-01: 3526 rows
   2025-06-01: 3021 rows
   2025-07-01: 2478 rows
   2025-08-01: 1985 rows
   2025-09-01: 1529 rows
   2025-10-01: 1041 rows
   2

✅ Saved Bronze table: /app/datamart/bronze/lms_loan_daily


In [8]:
process_silver_tables(spark)

Processing Silver table: features_attributes...


✅ Saved Silver table: /app/datamart/silver/features_attributes
Processing Silver table: features_financials...


✅ Saved Silver table: /app/datamart/silver/features_financials
Processing Silver table: feature_clickstream...


✅ Saved Silver table: /app/datamart/silver/feature_clickstream
Processing Silver table: lms_loan_daily...


✅ Saved Silver table: /app/datamart/silver/lms_loan_daily


In [9]:
splits = process_gold_tables(spark)

✅ No unexpected loan types found in data.


✅ No unexpected loan types found in data.


✅ No unexpected loan types found in data.
✅ No unexpected loan types found in data.


📊 TRAIN split: features shape=(7432, 119), labels shape=(7432, 3)


✅ train split saved.


📊 VAL split: features shape=(1541, 119), labels shape=(1541, 3)


✅ val split saved.


📊 TEST split: features shape=(1459, 119), labels shape=(1459, 3)


✅ test split saved.
📊 OOT split: features shape=(526, 119), labels shape=(526, 3)


✅ OOT split saved.


## Check output

In [10]:
splits

{'train': (DataFrame[Customer_ID: string, loan_start_date: date, Age: double, Age_valid: int, Annual_Income: float, Monthly_Inhand_Salary: float, Num_Bank_Accounts: double, Num_Credit_Card: int, Interest_Rate: int, Num_of_Loan: int, Delay_from_due_date: int, Num_of_Delayed_Payment: int, Changed_Credit_Limit: float, Num_Credit_Inquiries: int, Outstanding_Debt: float, Credit_Utilization_Ratio: float, Credit_History_Age_Months: int, Amount_invested_monthly: float, Monthly_Balance: float, avg_fe_1_L3M: double, avg_fe_2_L3M: double, avg_fe_3_L3M: double, avg_fe_4_L3M: double, avg_fe_5_L3M: double, avg_fe_6_L3M: double, avg_fe_7_L3M: double, avg_fe_8_L3M: double, avg_fe_9_L3M: double, avg_fe_10_L3M: double, avg_fe_11_L3M: double, avg_fe_12_L3M: double, avg_fe_13_L3M: double, avg_fe_14_L3M: double, avg_fe_15_L3M: double, avg_fe_16_L3M: double, avg_fe_17_L3M: double, avg_fe_18_L3M: double, avg_fe_19_L3M: double, avg_fe_20_L3M: double, sum_fe_1_L3M: double, sum_fe_2_L3M: double, sum_fe_3_L3M: d

In [13]:
X_train, y_train = splits['train']

In [14]:
x_df = X_train.toPandas()

In [15]:
len(x_df)

7432

In [20]:
x_df.columns.to_list()

['Customer_ID',
 'loan_start_date',
 'Age',
 'Age_valid',
 'Annual_Income',
 'Monthly_Inhand_Salary',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Num_of_Loan',
 'Delay_from_due_date',
 'Num_of_Delayed_Payment',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Outstanding_Debt',
 'Credit_Utilization_Ratio',
 'Credit_History_Age_Months',
 'Amount_invested_monthly',
 'Monthly_Balance',
 'avg_fe_1_L3M',
 'avg_fe_2_L3M',
 'avg_fe_3_L3M',
 'avg_fe_4_L3M',
 'avg_fe_5_L3M',
 'avg_fe_6_L3M',
 'avg_fe_7_L3M',
 'avg_fe_8_L3M',
 'avg_fe_9_L3M',
 'avg_fe_10_L3M',
 'avg_fe_11_L3M',
 'avg_fe_12_L3M',
 'avg_fe_13_L3M',
 'avg_fe_14_L3M',
 'avg_fe_15_L3M',
 'avg_fe_16_L3M',
 'avg_fe_17_L3M',
 'avg_fe_18_L3M',
 'avg_fe_19_L3M',
 'avg_fe_20_L3M',
 'sum_fe_1_L3M',
 'sum_fe_2_L3M',
 'sum_fe_3_L3M',
 'sum_fe_4_L3M',
 'sum_fe_5_L3M',
 'sum_fe_6_L3M',
 'sum_fe_7_L3M',
 'sum_fe_8_L3M',
 'sum_fe_9_L3M',
 'sum_fe_10_L3M',
 'sum_fe_11_L3M',
 'sum_fe_12_L3M',
 'sum_fe_13_L3M',
 'sum_fe_14_L3M',

In [17]:
y_df = y_train.toPandas()

In [18]:
len(y_df)

7432

In [19]:
def check_nulls(df, name="DataFrame"):
    from pyspark.sql import functions as F

    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ])
    
    print(f"\n🔎 Null check for {name}:")
    null_counts.show(truncate=False)
    return null_counts

check_nulls(X_train, "Feature Store (X)")
check_nulls(y_train, "Label Store (y)")



🔎 Null check for Feature Store (X):


+-----------+---------------+---+---------+-------------+---------------------+-----------------+---------------+-------------+-----------+-------------------+----------------------+--------------------+--------------------+----------------+------------------------+-------------------------+-----------------------+---------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+--------+--------+--------+--------+--------+

[Stage 375:=========================>                              (6 + 7) / 13]

+-----------+---------------+------------+
|Customer_ID|loan_start_date|default_flag|
+-----------+---------------+------------+
|0          |0              |0           |
+-----------+---------------+------------+



DataFrame[Customer_ID: bigint, loan_start_date: bigint, default_flag: bigint]

In [12]:
X_val, y_val = splits['val']

In [13]:
X_val_df = X_val.toPandas()

In [17]:
filtered_df = X_val_df[X_val_df['Customer_ID']=='CUS_0x10ac']

In [18]:
# Select columns ending with 'L3M' or 'LM'
cols_to_check = [c for c in filtered_df.columns if c.endswith('L3M') or c.endswith('LM')]

# Check if any values in these columns are non-zero
non_zero_mask = (filtered_df[cols_to_check] != 0).any(axis=1)

# Filter rows where any of these columns are not zero
rows_with_non_zero = filtered_df[non_zero_mask]

print(rows_with_non_zero)


Empty DataFrame
Columns: [Customer_ID, loan_start_date, Age, Age_valid, Annual_Income, Monthly_Inhand_Salary, Num_Bank_Accounts, Num_Credit_Card, Interest_Rate, Num_of_Loan, Delay_from_due_date, Num_of_Delayed_Payment, Changed_Credit_Limit, Num_Credit_Inquiries, Outstanding_Debt, Credit_Utilization_Ratio, Credit_History_Age_Months, Amount_invested_monthly, Monthly_Balance, avg_fe_1_L3M, avg_fe_2_L3M, avg_fe_3_L3M, avg_fe_4_L3M, avg_fe_5_L3M, avg_fe_6_L3M, avg_fe_7_L3M, avg_fe_8_L3M, avg_fe_9_L3M, avg_fe_10_L3M, avg_fe_11_L3M, avg_fe_12_L3M, avg_fe_13_L3M, avg_fe_14_L3M, avg_fe_15_L3M, avg_fe_16_L3M, avg_fe_17_L3M, avg_fe_18_L3M, avg_fe_19_L3M, avg_fe_20_L3M, sum_fe_1_L3M, sum_fe_2_L3M, sum_fe_3_L3M, sum_fe_4_L3M, sum_fe_5_L3M, sum_fe_6_L3M, sum_fe_7_L3M, sum_fe_8_L3M, sum_fe_9_L3M, sum_fe_10_L3M, sum_fe_11_L3M, sum_fe_12_L3M, sum_fe_13_L3M, sum_fe_14_L3M, sum_fe_15_L3M, sum_fe_16_L3M, sum_fe_17_L3M, sum_fe_18_L3M, sum_fe_19_L3M, sum_fe_20_L3M, fe_1_LM, fe_2_LM, fe_3_LM, fe_4_LM, fe_5_L

In [22]:
attributes_df = spark.read.parquet(silver_dir+"/features_attributes")

In [23]:
attr_df = attributes_df.toPandas()

In [24]:
attr_df

,Customer_ID,Name,Age,SSN,Occupation,Age_valid,snapshot_date
0,CUS_0x10ac,Zhouy,29.0,780-50-4730,Developer,1,2024-08-01
1,CUS_0x10c5,Moony,24.0,041-74-6785,None,1,2024-08-01
2,CUS_0x1145,Blenkinsopr,24.0,426-31-9194,Teacher,1,2024-08-01
3,CUS_0x11ac,Liana B.v,26.0,835-92-7751,Journalist,1,2024-08-01
4,CUS_0x122c,Papadimasf,48.0,883-73-9594,Entrepreneur,1,2024-08-01
...,...,...,...,...,...,...,...
12495,CUS_0xdf6,Euan Rochaa,55.0,155-73-3803,Mechanic,1,2023-09-01
12496,CUS_0xe23,Soyoung Kimg,39.0,825-49-2383,Musician,1,2023-09-01
12497,CUS_0xe4e,Mirwaisd,22.0,952-71-5963,Scientist,1,2023-09-01
12498,CUS_0xedd,Lauren Tarao,NaN,888-82-5609,Engineer,0,2023-09-01


In [25]:
df_attributes = spark.read.parquet(f"{silver_dir}/features_attributes")
df_attributes.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Age_valid: integer (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [26]:
null_counts = df_attributes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_attributes.columns
])

null_counts.show()

+-----------+----+---+---+----------+---------+-------------+
|Customer_ID|Name|Age|SSN|Occupation|Age_valid|snapshot_date|
+-----------+----+---+---+----------+---------+-------------+
|          0|   0|104|703|       880|        0|            0|
+-----------+----+---+---+----------+---------+-------------+



In [27]:
df_attr = df_attributes.toPandas()

In [28]:
df_attr['Occupation'].value_counts()

Occupation
Lawyer           828
Architect        795
Engineer         793
Accountant       791
Scientist        789
Teacher          782
Developer        780
Media_Manager    780
Mechanic         780
Entrepreneur     776
Journalist       761
Doctor           760
Musician         741
Manager          736
Writer           728
Name: count, dtype: int64

In [29]:
df_attr['Age'].value_counts()

Age
32.0     385
39.0     385
26.0     379
35.0     371
28.0     363
38.0     362
29.0     359
20.0     358
36.0     357
44.0     356
41.0     355
19.0     353
25.0     351
37.0     349
30.0     348
31.0     348
22.0     346
27.0     344
24.0     343
34.0     338
45.0     338
21.0     335
43.0     331
42.0     327
33.0     321
23.0     318
40.0     307
46.0     258
18.0     241
120.0    215
17.0     198
15.0     195
16.0     189
55.0     174
49.0     170
53.0     170
54.0     166
48.0     166
50.0     166
52.0     162
51.0     161
47.0     160
56.0      91
14.0      87
Name: count, dtype: int64

In [30]:
df_financials = spark.read.parquet(f"{silver_dir}/features_financials")
df_financials.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: float (nullable = true)
 |-- Monthly_Inhand_Salary: float (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: integer (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: integer (nullable = true)
 |-- Changed_Credit_Limit: float (nullable = true)
 |-- Num_Credit_Inquiries: integer (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: float (nullable = true)
 |-- Credit_Utilization_Ratio: float (nullable = true)
 |-- Credit_History_Age: string (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: float (nullable = true)
 |-- Amount_invested_monthly: float (nullable = true)
 |-- Payment_Behaviour: string (nullable = true)
 |-- Monthly_

In [31]:
null_counts = df_financials.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_financials.columns
])

null_counts.show()

+-----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+------------------+---------------------+-------------------+-----------------------+-----------------+---------------+--------------------+---------------------+-------------------------+-----------------+-------------+
|Customer_ID|Annual_Income|Monthly_Inhand_Salary|Num_Bank_Accounts|Num_Credit_Card|Interest_Rate|Num_of_Loan|Type_of_Loan|Delay_from_due_date|Num_of_Delayed_Payment|Changed_Credit_Limit|Num_Credit_Inquiries|Credit_Mix|Outstanding_Debt|Credit_Utilization_Ratio|Credit_History_Age|Payment_of_Min_Amount|Total_EMI_per_month|Amount_invested_monthly|Payment_Behaviour|Monthly_Balance|Credit_History_Years|Credit_History_Months|Credit_History_Age_Months|Type_of_Loan_list|snapshot_date|
+-----------+-------------+-----------

In [32]:
# Compute null counts per column
null_counts = df_financials.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_financials.columns
])

# Convert to a row, then flatten to key-value pairs
null_counts_long = (
    null_counts
    .toPandas()  # easier to reshape
    .T.reset_index()
    .rename(columns={"index": "column", 0: "null_count"})
)

# Filter only columns with nulls
null_counts_filtered = null_counts_long[null_counts_long["null_count"] > 0]

print(null_counts_filtered)


                  column  null_count
3      Num_Bank_Accounts           4
7           Type_of_Loan        1426
10  Changed_Credit_Limit         254
12            Credit_Mix        2611
19     Payment_Behaviour         998
20       Monthly_Balance           1


In [33]:
df_fin = df_financials.toPandas()

In [34]:
df_fin['Payment_of_Min_Amount'].value_counts()

Payment_of_Min_Amount
Yes    6571
No     4491
NM     1438
Name: count, dtype: int64

In [35]:
df_fin['Type_of_Loan'].value_counts(dropna=False)

Type_of_Loan
None                                                                                                       1426
Not Specified                                                                                               176
Credit-Builder Loan                                                                                         160
Personal Loan                                                                                               159
Debt Consolidation Loan                                                                                     158
                                                                                                           ... 
Personal Loan, Credit-Builder Loan, Payday Loan, and Debt Consolidation Loan                                  1
Not Specified, Student Loan, Not Specified, Mortgage Loan, Home Equity Loan, and Payday Loan                  1
Personal Loan, Personal Loan, Credit-Builder Loan, and Credit-Builder Loan                 

In [36]:
df_fin[['Type_of_Loan', 'Num_of_Loan']]

,Type_of_Loan,Num_of_Loan
0,"Credit-Builder Loan, Credit-Builder Loan, Home...",4
1,Payday Loan,1
2,"Student Loan, Payday Loan, Not Specified, Mort...",9
3,None,0
4,"Credit-Builder Loan, Home Equity Loan, and Aut...",3
...,...,...
12495,Student Loan,1
12496,"Not Specified, Auto Loan, and Mortgage Loan",3
12497,"Auto Loan, Personal Loan, Student Loan, Person...",7
12498,"Debt Consolidation Loan, Home Equity Loan, Per...",7


In [37]:
# Explode the array into rows, drop nulls, get distinct loan types
loan_types = (
    df_financials
    .withColumn("loan_type", F.explode_outer("Type_of_Loan_list"))
    .select("loan_type")
    .where(F.col("loan_type").isNotNull())
    .distinct()
    .orderBy("loan_type")
)

loan_types.show(truncate=False)

+-----------------------+
|loan_type              |
+-----------------------+
|Auto Loan              |
|Credit-Builder Loan    |
|Debt Consolidation Loan|
|Home Equity Loan       |
|Mortgage Loan          |
|Not Specified          |
|Payday Loan            |
|Personal Loan          |
|Student Loan           |
+-----------------------+



In [38]:
loan_types_list = [row["loan_type"] for row in loan_types.collect()]
loan_types_list

['Auto Loan',
 'Credit-Builder Loan',
 'Debt Consolidation Loan',
 'Home Equity Loan',
 'Mortgage Loan',
 'Not Specified',
 'Payday Loan',
 'Personal Loan',
 'Student Loan']

In [39]:
df_clicks = spark.read.parquet(f"{silver_dir}/feature_clickstream")
df_clicks.printSchema()

root
 |-- fe_1: float (nullable = true)
 |-- fe_2: float (nullable = true)
 |-- fe_3: float (nullable = true)
 |-- fe_4: float (nullable = true)
 |-- fe_5: float (nullable = true)
 |-- fe_6: float (nullable = true)
 |-- fe_7: float (nullable = true)
 |-- fe_8: float (nullable = true)
 |-- fe_9: float (nullable = true)
 |-- fe_10: float (nullable = true)
 |-- fe_11: float (nullable = true)
 |-- fe_12: float (nullable = true)
 |-- fe_13: float (nullable = true)
 |-- fe_14: float (nullable = true)
 |-- fe_15: float (nullable = true)
 |-- fe_16: float (nullable = true)
 |-- fe_17: float (nullable = true)
 |-- fe_18: float (nullable = true)
 |-- fe_19: float (nullable = true)
 |-- fe_20: float (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [41]:
clicks = df_clicks.toPandas()

In [328]:
df_loan = spark.read.parquet(f"{silver_dir}/lms_loan_daily")
df_loan.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- loan_start_date: date (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- installment_num: integer (nullable = true)
 |-- loan_amt: float (nullable = true)
 |-- due_amt: float (nullable = true)
 |-- paid_amt: float (nullable = true)
 |-- overdue_amt: float (nullable = true)
 |-- balance: float (nullable = true)
 |-- snapshot_date: date (nullable = true)



In [44]:
df_fin[~df_fin['Customer_ID'].isin(clicks['Customer_ID'])]['snapshot_date'].value_counts()

snapshot_date
2024-08-01    543
2025-01-01    526
2024-12-01    515
2024-07-01    505
2024-09-01    493
2024-11-01    488
2024-10-01    456
Name: count, dtype: int64

## Bronze

In [3]:
def process_bronze_tables(raw_data_dir: str, bronze_dir: str, spark: SparkSession):
    """
    Process raw CSVs into Bronze Parquet tables, partitioned by snapshot_date.

    Args:
        raw_data_dir (str): Path to folder containing the raw CSVs.
        bronze_dir (str): Base path to save Bronze tables.
        spark (SparkSession): Active Spark session.
    """

    # Map of raw filenames -> bronze table names
    datasets = {
        "features_attributes.csv": "features_attributes",
        "features_financials.csv": "features_financials",
        "feature_clickstream.csv": "feature_clickstream",
        "lms_loan_daily.csv": "lms_loan_daily"
    }

    for filename, table_name in datasets.items():
        csv_path = f"{raw_data_dir}/{filename}"

        print(f"Processing {filename}...")

        # Load CSV
        df = spark.read.csv(csv_path, header=True, inferSchema=True)

        # Ensure snapshot_date is DateType
        df = df.withColumn("snapshot_date", to_date(col("snapshot_date")))

        # Show row count per snapshot_date
        counts = df.groupBy("snapshot_date").count().orderBy("snapshot_date")
        print(f"📊 Row counts per snapshot_date for {table_name}:")
        for row in counts.collect():
            print(f"   {row['snapshot_date']}: {row['count']} rows")
        
        # Save to Bronze as partitioned Parquet (1 file per snapshot_date)
        output_path = f"{bronze_dir}/{table_name}"
        (
            df
            .coalesce(1)  # force 1 file per snapshot_date partition
            .write
            .mode("overwrite")
            .partitionBy("snapshot_date")
            .parquet(output_path)
        )

        print(f"✅ Saved Bronze table: {output_path}")



## EDA 2

In [ ]:
lms_loan_daily_df
clickstream_df
attributes_df
financials_df

In [17]:
def check_null(df):
    for col in df.columns:
        if df[col].isnull().any():
            print(f"{col} contains missing values")

In [22]:
check_null(financials_df)

Type_of_Loan contains missing values


In [25]:
attributes_df.describe()

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
count,12500,12500,12500,12500,12500,12500
unique,12500,10139,301,11798,16,25
top,CUS_0xffd,Langep,26,#F%$D@*&8,_______,2024-08-01
freq,1,6,364,703,880,543


In [296]:
attributes_df[~attributes_df['Age'].str.isnumeric()]

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
10,CUS_0x1032,Wahbap,40_,620-58-8045,Lawyer,2023-08-01
22,CUS_0x1057,David Sheppardv,46_,741-07-3912,Writer,2023-09-01
50,CUS_0x10e7,Carewj,3843_,094-89-6380,_______,2024-11-01
52,CUS_0x10ee,Hudsonb,30_,699-80-2426,Journalist,2023-06-01
62,CUS_0x111c,Deepaa,24_,420-66-8144,Journalist,2024-03-01
...,...,...,...,...,...,...
12431,CUS_0xedd,Lauren Tarao,-500,888-82-5609,Engineer,2023-09-01
12456,CUS_0xf58,Tiisetso Motsoenengj,29_,108-83-3774,Scientist,2023-04-01
12468,CUS_0xf8d,Jennifer Ablanr,55_,712-91-8519,Engineer,2023-04-01
12471,CUS_0xf95,Alexei Anishchukz,17_,082-42-3634,Entrepreneur,2025-01-01


In [38]:
filtered_df = attributes_df[~attributes_df['Customer_ID'].astype(str).str.startswith('CUS_')]
filtered_df

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date


In [ ]:
non_numeric = attributes_df[~attributes_df['Customer_ID'].astype(str).str.isnumeric()]
print(non_numeric['Customer_ID'].unique())

In [41]:
pattern = r'^\d{3}-\d{2}-\d{4}$'
filtered_df = attributes_df[~attributes_df['SSN'].astype(str).str.match(pattern)]
filtered_df

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
2,CUS_0x100b,Shirboni,19,#F%$D@*&8,Media_Manager,2024-03-01
9,CUS_0x102e,Rhysn,26,#F%$D@*&8,Scientist,2024-04-01
15,CUS_0x1044,Maki Shirakip,44,#F%$D@*&8,_______,2023-06-01
17,CUS_0x104a,Leahk,37,#F%$D@*&8,Mechanic,2023-12-01
23,CUS_0x105b,Patricia Zengerlel,24,#F%$D@*&8,Accountant,2023-11-01
...,...,...,...,...,...,...
12409,CUS_0xe94,Nishanta,53,#F%$D@*&8,Musician,2024-03-01
12425,CUS_0xecc,Papadimase,25,#F%$D@*&8,Lawyer,2024-11-01
12442,CUS_0xf20,Anurag Kotokyf,41,#F%$D@*&8,Musician,2023-04-01
12446,CUS_0xf30,Hornbyb,29,#F%$D@*&8,Journalist,2023-06-01


Some records contain junk values in SSN 

In [45]:
filtered_df = attributes_df[~attributes_df['Occupation'].astype(str).str.match(r'^[A-Za-z\s]+$')]
filtered_df[filtered_df['Occupation']!='Media_Manager']

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
15,CUS_0x1044,Maki Shirakip,44,#F%$D@*&8,_______,2023-06-01
19,CUS_0x104f,Markm,20,264-84-8069,_______,2024-10-01
34,CUS_0x1098,Krudyz,23,466-10-5742,_______,2025-01-01
45,CUS_0x10c5,Moony,24,041-74-6785,_______,2024-08-01
50,CUS_0x10e7,Carewj,3843_,094-89-6380,_______,2024-11-01
...,...,...,...,...,...,...
12412,CUS_0xea5,erniee,28,196-98-4126,_______,2024-06-01
12424,CUS_0xecb,Junkou,21,899-79-3678,_______,2025-01-01
12452,CUS_0xf4b,Lu Jianxink,30,527-06-9414,_______,2024-05-01
12486,CUS_0xfcc,Suvashree Deyy,22,388-13-4625,_______,2023-02-01


Empty fields in Occupation is represented with '_______'

In [331]:
clickstream_df.describe(include='all')

,fe_1,fe_2,fe_3,fe_4,fe_5,fe_6,fe_7,fe_8,fe_9,fe_10,...,fe_13,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,Customer_ID,snapshot_date
count,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,...,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376.000000,215376,215376
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8974,24
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CUS_0xfe4,2023-01-01
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24,8974
mean,101.414796,103.096195,104.333709,105.648503,106.996676,103.235922,107.070337,110.718724,114.406354,117.775797,...,100.420804,100.025801,99.600383,99.860834,100.129425,100.125139,100.341162,100.046259,NaN,NaN
std,99.833594,99.930002,100.599865,100.326065,100.693607,100.270388,100.323265,100.243698,100.186139,100.807686,...,100.881826,101.070371,101.145906,100.751876,101.298428,102.231587,102.666804,103.589358,NaN,NaN
min,-378.000000,-356.000000,-399.000000,-307.000000,-343.000000,-321.000000,-368.000000,-361.000000,-328.000000,-317.000000,...,-355.000000,-394.000000,-351.000000,-342.000000,-329.000000,-344.000000,-401.000000,-354.000000,NaN,NaN
25%,34.000000,36.000000,36.000000,38.000000,39.000000,36.000000,39.000000,43.000000,47.000000,50.000000,...,33.000000,32.000000,31.000000,32.000000,32.000000,31.000000,31.000000,30.000000,NaN,NaN
50%,102.000000,103.000000,104.000000,106.000000,107.000000,103.000000,107.000000,111.000000,115.000000,118.000000,...,101.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,NaN,NaN
75%,169.000000,171.000000,172.000000,173.000000,175.000000,171.000000,174.000000,179.000000,182.000000,186.000000,...,168.000000,168.000000,168.000000,168.000000,169.000000,169.000000,170.000000,170.000000,NaN,NaN


In [55]:
financials_df.describe(include='all')

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
count,12500,12500,12500.000000,12500.000000,12500.000000,12500.000000,12500,11074,12500.000000,12500,...,12500,12500,12500.000000,12500,12500,12500.000000,12500,12500,12500,12500
unique,12500,12492,NaN,NaN,NaN,NaN,87,6260,NaN,168,...,4,12213,NaN,393,3,NaN,11921,7,12500,25
top,CUS_0xffd,17273.83,NaN,NaN,NaN,NaN,3,Not Specified,NaN,19,...,Standard,1360.45,NaN,16 Years and 2 Months,Yes,NaN,__10000__,Low_spent_Small_value_payments,389.434630761722,2024-08-01
freq,1,2,NaN,NaN,NaN,NaN,1808,176,NaN,735,...,4497,3,NaN,79,6571,NaN,558,3202,1,543
mean,NaN,NaN,4188.592303,16.939920,23.172720,73.213360,NaN,NaN,21.060880,NaN,...,NaN,NaN,32.349265,NaN,NaN,1488.394291,NaN,NaN,NaN,NaN
std,NaN,NaN,3180.147611,114.350815,132.005866,468.682227,NaN,NaN,14.863091,NaN,...,NaN,NaN,5.156815,NaN,NaN,8561.449910,NaN,NaN,NaN,NaN
min,NaN,NaN,303.645417,-1.000000,0.000000,1.000000,NaN,NaN,-5.000000,NaN,...,NaN,NaN,20.100770,NaN,NaN,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,1624.937917,3.000000,4.000000,8.000000,NaN,NaN,10.000000,NaN,...,NaN,NaN,28.066517,NaN,NaN,31.496968,NaN,NaN,NaN,NaN
50%,NaN,NaN,3087.595000,6.000000,5.000000,14.000000,NaN,NaN,18.000000,NaN,...,NaN,NaN,32.418953,NaN,NaN,72.887628,NaN,NaN,NaN,NaN
75%,NaN,NaN,5947.364167,7.000000,7.000000,20.000000,NaN,NaN,28.000000,NaN,...,NaN,NaN,36.623650,NaN,NaN,169.634826,NaN,NaN,NaN,NaN


In [54]:
financials_df.dtypes

Customer_ID                  object
Annual_Income                object
Monthly_Inhand_Salary       float64
Num_Bank_Accounts             int64
Num_Credit_Card               int64
Interest_Rate                 int64
Num_of_Loan                  object
Type_of_Loan                 object
Delay_from_due_date           int64
Num_of_Delayed_Payment       object
Changed_Credit_Limit         object
Num_Credit_Inquiries        float64
Credit_Mix                   object
Outstanding_Debt             object
Credit_Utilization_Ratio    float64
Credit_History_Age           object
Payment_of_Min_Amount        object
Total_EMI_per_month         float64
Amount_invested_monthly      object
Payment_Behaviour            object
Monthly_Balance              object
snapshot_date                object
dtype: object

In [56]:
financials_df[~financials_df['Annual_Income'].astype(str).str.replace('.', '', 1).str.isnumeric()]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
1,CUS_0x1009,52312.68_,4250.390000,6,5,17,4,"Not Specified, Home Equity Loan, Credit-Builde...",5,18,...,_,202.68,40.286997,31 Years and 0 Months,Yes,108.366467,58.66019164829086,High_spent_Medium_value_payments,508.01234122645366,2025-01-01
29,CUS_0x107c,49718.55_,4179.212500,7,10,27,6_,"Payday Loan, Mortgage Loan, Home Equity Loan, ...",27,17,...,Bad,1920.25,30.109638,14 Years and 0 Months,Yes,231.956781,430.88393873056435,Low_spent_Small_value_payments,45.080530624405014,2023-11-01
34,CUS_0x1098,20652.98_,1468.081667,7,5,21,2,"Auto Loan, and Payday Loan",24,10,...,Standard,1859.53,37.441101,18 Years and 11 Months,NM,33.238356,54.60338563277085,Low_spent_Small_value_payments,348.96642479393626,2025-01-01
51,CUS_0x10eb,28315.95_,2415.662500,4,7,10,2_,"Student Loan, and Payday Loan",8,8,...,Good,677.4,26.912323,20 Years and 5 Months,No,30.230996,133.04883496511272,Low_spent_Small_value_payments,368.28641876324724,2023-03-01
56,CUS_0x1100,43062.54_,3607.545000,6,10,23,2,"Mortgage Loan, and Student Loan",55,20,...,Standard,2169.75,30.042623,15 Years and 3 Months,Yes,47.499819,71.94658528758634,High_spent_Medium_value_payments,491.30809597080133,2023-10-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12455,CUS_0xf55,78443.48_,6358.956667,7,5,23,4,"Personal Loan, Home Equity Loan, Mortgage Loan...",39,19,...,Bad,1527.77,24.704429,15 Years and 10 Months,NM,177.387563,528.7469053018515,Low_spent_Medium_value_payments,209.76119880079318,2024-12-01
12468,CUS_0xf8d,43983.52_,3722.293333,0,5,5,4,"Debt Consolidation Loan, Student Loan, Not Spe...",5,1,...,Good,690.43,26.997008,18 Years and 0 Months,No,45104.000000,331.65131234867863,Low_spent_Large_value_payments,171.30558022946562,2023-04-01
12469,CUS_0xf8f,68236.12_,5542.343333,7,10,34,5,"Student Loan, Personal Loan, Auto Loan, Debt C...",29,12,...,Standard,2535.12,28.793494,16 Years and 5 Months,Yes,232.555758,490.08391668712824,!@9#%8,101.59465825623352,2023-07-01
12484,CUS_0xfc9,17290.42_,1426.868333,6,4,31,7,"Payday Loan, Student Loan, Student Loan, Debt ...",2,14,...,Standard,1594.66,25.858708,14 Years and 2 Months,Yes,65.017829,124.7647062720153,Low_spent_Small_value_payments,242.9042980787336,2023-01-01


In [57]:
financials_df[financials_df['Annual_Income'].astype(str).str.endswith('_')]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
1,CUS_0x1009,52312.68_,4250.390000,6,5,17,4,"Not Specified, Home Equity Loan, Credit-Builde...",5,18,...,_,202.68,40.286997,31 Years and 0 Months,Yes,108.366467,58.66019164829086,High_spent_Medium_value_payments,508.01234122645366,2025-01-01
29,CUS_0x107c,49718.55_,4179.212500,7,10,27,6_,"Payday Loan, Mortgage Loan, Home Equity Loan, ...",27,17,...,Bad,1920.25,30.109638,14 Years and 0 Months,Yes,231.956781,430.88393873056435,Low_spent_Small_value_payments,45.080530624405014,2023-11-01
34,CUS_0x1098,20652.98_,1468.081667,7,5,21,2,"Auto Loan, and Payday Loan",24,10,...,Standard,1859.53,37.441101,18 Years and 11 Months,NM,33.238356,54.60338563277085,Low_spent_Small_value_payments,348.96642479393626,2025-01-01
51,CUS_0x10eb,28315.95_,2415.662500,4,7,10,2_,"Student Loan, and Payday Loan",8,8,...,Good,677.4,26.912323,20 Years and 5 Months,No,30.230996,133.04883496511272,Low_spent_Small_value_payments,368.28641876324724,2023-03-01
56,CUS_0x1100,43062.54_,3607.545000,6,10,23,2,"Mortgage Loan, and Student Loan",55,20,...,Standard,2169.75,30.042623,15 Years and 3 Months,Yes,47.499819,71.94658528758634,High_spent_Medium_value_payments,491.30809597080133,2023-10-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12455,CUS_0xf55,78443.48_,6358.956667,7,5,23,4,"Personal Loan, Home Equity Loan, Mortgage Loan...",39,19,...,Bad,1527.77,24.704429,15 Years and 10 Months,NM,177.387563,528.7469053018515,Low_spent_Medium_value_payments,209.76119880079318,2024-12-01
12468,CUS_0xf8d,43983.52_,3722.293333,0,5,5,4,"Debt Consolidation Loan, Student Loan, Not Spe...",5,1,...,Good,690.43,26.997008,18 Years and 0 Months,No,45104.000000,331.65131234867863,Low_spent_Large_value_payments,171.30558022946562,2023-04-01
12469,CUS_0xf8f,68236.12_,5542.343333,7,10,34,5,"Student Loan, Personal Loan, Auto Loan, Debt C...",29,12,...,Standard,2535.12,28.793494,16 Years and 5 Months,Yes,232.555758,490.08391668712824,!@9#%8,101.59465825623352,2023-07-01
12484,CUS_0xfc9,17290.42_,1426.868333,6,4,31,7,"Payday Loan, Student Loan, Student Loan, Debt ...",2,14,...,Standard,1594.66,25.858708,14 Years and 2 Months,Yes,65.017829,124.7647062720153,Low_spent_Small_value_payments,242.9042980787336,2023-01-01


In [300]:
financials_df['Num_Bank_Accounts'].describe()

count    12500.000000
mean        16.939920
std        114.350815
min         -1.000000
25%          3.000000
50%          6.000000
75%          7.000000
max       1756.000000
Name: Num_Bank_Accounts, dtype: float64

In [313]:
financials_df[financials_df['Num_Bank_Accounts']==11]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
6200,CUS_0x6cd1,81338.16,6642.1800,11,10,33,8,"Credit-Builder Loan, Mortgage Loan, Mortgage L...",60,15,...,_,4455.19,28.261134,14 Years and 5 Months,Yes,356.766533,441.52624142297316,Low_spent_Large_value_payments,135.925226045072,2023-02-01
7580,CUS_0x81c5,55554.69,4649.5575,11,9,34,7,"Personal Loan, Home Equity Loan, Home Equity L...",37,17,...,Bad,4974.31,38.149970,1 Years and 5 Months,Yes,228.116758,__10000__,High_spent_Medium_value_payments,358.5184199830938,2023-07-01
10017,CUS_0xa59,20961.27,1833.7725,11,11,16,3,"Home Equity Loan, Not Specified, and Payday Loan",44,18,...,Standard,1917.28,30.238067,16 Years and 6 Months,Yes,43.215031,69.85557031743606,!@9#%8,320.30664901229267,2023-03-01


In [59]:
financials_df[financials_df['Num_Bank_Accounts']<0]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
3330,CUS_0x43bc,22312.07,2013.339167,-1,3,4,3,"Home Equity Loan, Debt Consolidation Loan, and...",9,8,...,Good,51.37,30.059211,23 Years and 10 Months,No,32.891186,23.00309350191133,High_spent_Medium_value_payments,395.43963679490616,2024-02-01
4161,CUS_0x4f2a,22136920.0,10434.146667,-1,6,1,2,"Auto Loan, and Not Specified",10,6,...,_,1151.7,38.278518,22 Years and 10 Months,No,196.587321,338.67230317776176,High_spent_Medium_value_payments,758.1550423229277,2024-11-01
4849,CUS_0x5993,30352.11,2317.342500,-1,4,7,1,Student Loan,8,4,...,Good,644.57,32.937399,24 Years and 0 Months,No,16.483566,89.62296249058659,High_spent_Medium_value_payments,375.62772128619685,2024-10-01
10204,CUS_0xa878,117851.07,9870.922500,-1,6,7,0,NaN,27,1,...,Good,607.78,34.041733,21 Years and 0 Months,No,0.000000,467.41582464616476,High_spent_Small_value_payments,779.6764253538355,2023-07-01


In [74]:
financials_df['Num_Credit_Card'].describe()

count    12500.000000
mean        23.172720
std        132.005866
min          0.000000
25%          4.000000
50%          5.000000
75%          7.000000
max       1499.000000
Name: Num_Credit_Card, dtype: float64

In [64]:
q1 = financials_df['Num_Credit_Card'].quantile(0.25)
q3 = financials_df['Num_Credit_Card'].quantile(0.75)
iqr = q3 - q1
cutoff = q3 + 1.5 * iqr
print(f"Suggested cut-off for outliers: {cutoff}")

Suggested cut-off for outliers: 11.5


In [75]:
financials_df[financials_df['Num_Credit_Card']==11]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
805,CUS_0x1d6f,17438.8,1237.233333,8,11,16,5,"Not Specified, Mortgage Loan, Credit-Builder L...",21,18,...,Bad,2635.91,30.585343,9 Years and 9 Months,Yes,44.235325,150.64119863743872,Low_spent_Small_value_payments,218.84680919993926,2023-06-01
2136,CUS_0x3187,9983.345,758.945417,7,11,15,5,"Not Specified, Student Loan, Auto Loan, Payday...",22,16,...,Standard,1823.35,26.294944,9 Years and 9 Months,Yes,29.241519,10.612907877548746,High_spent_Large_value_payments,276.0401150994764,2024-01-01
3988,CUS_0x4d0,50050.56,4406.880000,10,11,22,9,"Home Equity Loan, Personal Loan, Mortgage Loan...",38,19,...,Bad,2913.45,34.975077,8 Years and 4 Months,Yes,334.601238,205.9493824472522,!@9#%8,190.1373798886116,2024-08-01
7784,CUS_0x84df,45595.08,3189.212103,9,11,27,6,"Credit-Builder Loan, Auto Loan, Home Equity Lo...",60,23,...,Bad,2723.1,32.212055,14 Years and 5 Months,Yes,491.961846,45.197789294981824,High_spent_Large_value_payments,411.57726208265814,2024-02-01
8932,CUS_0x958b,57700.16,4864.346667,8,11,22,5,"Mortgage Loan, Home Equity Loan, Auto Loan, No...",59,21,...,Bad,1728.05,24.919244,19 Years and 0 Months,Yes,221.763278,__10000__,Low_spent_Small_value_payments,446.7448842837021,2023-06-01
9545,CUS_0x9ea6,49650.36,3995.530000,7,11,26,6,"Not Specified, Personal Loan, Credit-Builder L...",28,10,...,Standard,1826.54,30.597357,14 Years and 3 Months,Yes,194.312441,324.0450546077384,Low_spent_Medium_value_payments,161.19550462991967,2024-09-01
10017,CUS_0xa59,20961.27,1833.772500,11,11,16,3,"Home Equity Loan, Not Specified, and Payday Loan",44,18,...,Standard,1917.28,30.238067,16 Years and 6 Months,Yes,43.215031,69.85557031743606,!@9#%8,320.30664901229267,2023-03-01
12418,CUS_0xebc,14310.25,1064.520833,10,11,34,9_,"Debt Consolidation Loan, Auto Loan, Credit-Bui...",15,26,...,Bad,2188.14,27.291220,12 Years and 6 Months,NM,92.414081,140.90193979075994,Low_spent_Small_value_payments,163.13606299905797,2023-03-01


In [80]:
financials_df['Interest_Rate'].describe()

count    12500.000000
mean        73.213360
std        468.682227
min          1.000000
25%          8.000000
50%         14.000000
75%         20.000000
max       5789.000000
Name: Interest_Rate, dtype: float64

In [76]:
q1 = financials_df['Interest_Rate'].quantile(0.25)
q3 = financials_df['Interest_Rate'].quantile(0.75)
iqr = q3 - q1
cutoff = q3 + 1.5 * iqr
print(f"Suggested cut-off for outliers: {cutoff}")

Suggested cut-off for outliers: 38.0


In [85]:
financials_df[financials_df['Interest_Rate']==34]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
63,CUS_0x1123,15264.98,1266.081667,6,5,34,3,"Home Equity Loan, Credit-Builder Loan, and Aut...",23,14,...,Bad,2375.88,24.094344,8 Years and 10 Months,Yes,50563.000000,136.95202265544256,Low_spent_Small_value_payments,242.8660473000196,2024-10-01
130,CUS_0x123a,36228.2,2991.016667,8,7,34,2,"Personal Loan, and Not Specified",28,15,...,_,1643.06,37.112483,18 Years and 9 Months,Yes,47.252408,176.60375699560146,Low_spent_Medium_value_payments,366.9782289080675,2023-03-01
299,CUS_0x14ce,17776.2,1727.350000,955,6,34,5,"Student Loan, Debt Consolidation Loan, Persona...",51,12,...,Standard,2221.13,26.029249,18 Years and 11 Months,Yes,57.275959,__10000__,Low_spent_Small_value_payments,346.8038665689816,2024-04-01
409,CUS_0x1698,9221.875,598.489583,6,5,34,4,"Auto Loan, Debt Consolidation Loan, Personal L...",40,16,...,Standard,2374.03,35.238513,16 Years and 4 Months,NM,28.987763,66.92892025117658,Low_spent_Small_value_payments,253.93227545815205,2023-06-01
551,CUS_0x1900,8137.625,680.135417,8,6,34,8,"Mortgage Loan, Mortgage Loan, Mortgage Loan, P...",35,21,...,Bad,3938.7,23.169472,2 Years and 5 Months,Yes,53.343671,35.01509474856865,Low_spent_Small_value_payments,269.6547756668612,2024-08-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12175,CUS_0xc5b2,60176.61,4989.717500,7,9,34,8,"Mortgage Loan, Auto Loan, Personal Loan, Perso...",25,21,...,Bad,4176.76,34.987376,12 Years and 5 Months,Yes,246.918669,190.33549746852384,High_spent_Small_value_payments,321.71758334680266,2023-02-01
12261,CUS_0xc715,32291.88_,2876.990000,7,10,34,5,"Home Equity Loan, Student Loan, Payday Loan, N...",26,21,...,Bad,3134.09,35.970080,1 Years and 11 Months,Yes,119.795327,181.4320921534653,High_spent_Small_value_payments,252.52401614602886,2023-07-01
12336,CUS_0xd83,19310.62,1463.093138,9,6,34,5,"Debt Consolidation Loan, Mortgage Loan, Studen...",47,10,...,Standard,1306.08,25.611673,10 Years and 3 Months,Yes,238.490291,__10000__,High_spent_Medium_value_payments,283.98976582702164,2023-06-01
12418,CUS_0xebc,14310.25,1064.520833,10,11,34,9_,"Debt Consolidation Loan, Auto Loan, Credit-Bui...",15,26,...,Bad,2188.14,27.291220,12 Years and 6 Months,NM,92.414081,140.90193979075994,Low_spent_Small_value_payments,163.13606299905797,2023-03-01


In [ ]:
financials_df[financials_df['Num_of_Loan']]

In [87]:
financials_df['Num_of_Loan'].describe()

count     12500
unique       87
top           3
freq       1808
Name: Num_of_Loan, dtype: object

In [95]:
financials_df[~financials_df['Num_of_Loan'].astype(str).str.isnumeric()]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
13,CUS_0x103e,98690.8,8262.233333,4,6,9,1_,Student Loan,6,17,...,Good,706.96,26.860663,26 Years and 11 Months,No,55.004408,913.4813186573292,Low_spent_Small_value_payments,147.7376071067124,2024-12-01
29,CUS_0x107c,49718.55_,4179.212500,7,10,27,6_,"Payday Loan, Mortgage Loan, Home Equity Loan, ...",27,17,...,Bad,1920.25,30.109638,14 Years and 0 Months,Yes,231.956781,430.88393873056435,Low_spent_Small_value_payments,45.080530624405014,2023-11-01
44,CUS_0x10c0,49454.13,4328.177500,8,5,23,2_,"Payday Loan, and Auto Loan",59,23,...,Bad,1388.56,28.978427,13 Years and 4 Months,NM,73.346359,148.5406575943259,Low_spent_Medium_value_payments,459.07807620141364,2024-02-01
50,CUS_0x10e7,64933.76,5368.146667,3,3,33,-100,"Mortgage Loan, Auto Loan, Home Equity Loan, an...",15,19,...,Standard,2699.17_,24.958638,14 Years and 4 Months,Yes,193.692582,186.14666535889234,High_spent_Medium_value_payments,406.9754192189962,2024-11-01
51,CUS_0x10eb,28315.95_,2415.662500,4,7,10,2_,"Student Loan, and Payday Loan",8,8,...,Good,677.4,26.912323,20 Years and 5 Months,No,30.230996,133.04883496511272,Low_spent_Small_value_payments,368.28641876324724,2023-03-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12451,CUS_0xf45,32057.3,2606.441667,9,8,16,7_,"Student Loan, Payday Loan, Home Equity Loan, H...",45,25,...,Bad,1327.26,38.598795,10 Years and 4 Months,Yes,164.859426,207.57708393239278,Low_spent_Small_value_payments,178.207656548737,2023-05-01
12480,CUS_0xfb4,13982.725,931.227083,7,6,12,4_,"Student Loan, Payday Loan, Mortgage Loan, and ...",12,14,...,Standard,1458.61,29.666479,30 Years and 2 Months,Yes,37.983363,25.80070520746518,High_spent_Small_value_payments,289.3386401834556,2023-12-01
12485,CUS_0xfcb,10805.56,627.463333,5,6,15,2_,"Not Specified, and Mortgage Loan",29,16,...,_,298.98,38.349012,18 Years and 0 Months,No,13.161860,36.90837609570188,Low_spent_Small_value_payments,302.67609724613584,2023-04-01
12490,CUS_0xfdf,70114.38,5679.865000,0,4,12,-100,"Home Equity Loan, Auto Loan, and Home Equity Loan",4,0,...,Good,918.89,40.434529,25 Years and 1 Months,No,131.472173,146.86000235186486,High_spent_Large_value_payments,529.6543248971085,2025-01-01


In [100]:
financials_df['Num_of_Loan'] = financials_df['Num_of_Loan'].astype(str).str.replace('_', '', regex=False)
financials_df['Num_of_Loan'] = financials_df['Num_of_Loan'].astype(int)
filtered_df = financials_df[financials_df['Num_of_Loan'].astype(float) >= 0]

In [101]:
filtered_df['Num_of_Loan'].describe()

count    12000.000000
mean         7.395250
std         62.880332
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max       1495.000000
Name: Num_of_Loan, dtype: float64

In [107]:
filtered_df[filtered_df['Num_of_Loan']==10]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date


In [214]:
financials_df['Type_of_Loan'].value_counts(dropna=False)

Type_of_Loan
NaN                                                                                                     1426
Not Specified                                                                                            176
Credit-Builder Loan                                                                                      160
Personal Loan                                                                                            159
Debt Consolidation Loan                                                                                  158
                                                                                                        ... 
Student Loan, Payday Loan, Credit-Builder Loan, and Student Loan                                           1
Student Loan, Home Equity Loan, Credit-Builder Loan, Debt Consolidation Loan, and Student Loan             1
Mortgage Loan, Home Equity Loan, Personal Loan, and Personal Loan                                          1
Debt C

In [217]:
financials_df['Type_of_Loan'].isna().sum()

np.int64(1426)

In [218]:
1426/12500

0.11408

In [111]:
financials_df[financials_df['Delay_from_due_date']<0]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
165,CUS_0x12be,15717.275,1131.034316,1,5,8,1,Credit-Builder Loan,-1,8,...,Good,1020.27,26.462115,31 Years and 5 Months,No,152.181475,146.84954628537986,Low_spent_Small_value_payments,257.78487022590286,2023-04-01
220,CUS_0x139b,29313.3,2151.775000,4,5,1,3,"Credit-Builder Loan, Home Equity Loan, and Not...",-1,4,...,Good,341.96,28.195616,32 Years and 5 Months,No,52.217020,128.96051975666418,High_spent_Small_value_payments,293.99995988095543,2024-08-01
435,CUS_0x1709,65945.7,5486.475000,5,7,8,0,NaN,-1,8,...,Good,215.05,31.352868,18 Years and 8 Months,No,0.000000,147.72194202875832,Low_spent_Large_value_payments,670.9255579712418,2023-08-01
457,CUS_0x178b,43456.5,3437.375000,2,680,8,0,NaN,-4,8,...,_,389.9,24.298378,31 Years and 5 Months,NM,0.000000,398.74744046917954,Low_spent_Small_value_payments,234.99005953082053,2024-02-01
952,CUS_0x1fbd,76881.84,6529.820000,3,7,6,1,Not Specified,-1,7,...,Good,257.87,41.712807,27 Years and 10 Months,No,61.440036,73.75544697431374,High_spent_Large_value_payments,757.7865165615817,2024-04-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12329,CUS_0xd6f,70045.84,5515.822143,4,6,6,2,"Personal Loan, and Mortgage Loan",-2,5,...,Good,1438.66,27.975537,19 Years and 4 Months,No,604.644000,72.46230202460849,High_spent_Large_value_payments,667.0402212070542,2023-01-01
12368,CUS_0xdef,158604.0,13273.000000,4,3,1,4,"Payday Loan, Student Loan, Home Equity Loan, a...",-1,9,...,Good,203.93,34.953265,32 Years and 11 Months,No,438.342387,294.14561023642483,High_spent_Large_value_payments,834.8120028043236,2025-01-01
12481,CUS_0xfb6,42165.91,3658.825833,4,3,7,0,NaN,-1,11,...,_,981.66,26.853329,30 Years and 8 Months,NM,0.000000,104.77549855671684,High_spent_Medium_value_payments,511.1070847766166,2023-04-01
12482,CUS_0xfb8,34636.64,2892.636024,3,1,9,4,"Credit-Builder Loan, Payday Loan, Debt Consoli...",-2,10,...,_,644.87,26.502110,24 Years and 0 Months,No,347.164715,60.492381650303116,High_spent_Large_value_payments,389.7322134561853,2023-07-01


In [252]:
financials_df['Delay_from_due_date'].describe()

count    12500.000000
mean        21.060880
std         14.863091
min         -5.000000
25%         10.000000
50%         18.000000
75%         28.000000
max         67.000000
Name: Delay_from_due_date, dtype: float64

In [245]:
financials_df['Num_of_Delayed_Payment'] = financials_df['Num_of_Delayed_Payment'].astype(str).str.replace('_', '', regex=False)
filtered_df = financials_df[financials_df['Num_of_Delayed_Payment'].astype(float) >= 0]

In [256]:
financials_df['Num_of_Delayed_Payment'].astype(float).describe()

count    12500.000000
mean        33.024960
std        238.695905
min         -3.000000
25%          9.000000
50%         14.000000
75%         18.000000
max       4293.000000
Name: Num_of_Delayed_Payment, dtype: float64

In [262]:
filtered_df[filtered_df['Num_of_Delayed_Payment'].astype(float)==28]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date
357,CUS_0x15d5,72207.52,5881.293333,6,7,21,5,"Home Equity Loan, Debt Consolidation Loan, Not...",44,28,...,Bad,1803.68,33.788071,13 Years and 4 Months,Yes,287.623792,284.32256410896457,High_spent_Small_value_payments,276.1829771605767,2024-03-01
958,CUS_0x1fd1,33494.36,2677.196667,9,9,28,8,"Debt Consolidation Loan, Home Equity Loan, Hom...",52,28,...,Bad,3304.83,39.158808,9 Years and 6 Months,Yes,197.255533,66.18393827658439,High_spent_Medium_value_payments,254.28019533787528,2024-05-01
2782,CUS_0x3bc6,14414.25,1351.187500,9,9,16,9,"Credit-Builder Loan, Home Equity Loan, Auto Lo...",47,28,...,Bad,2558.21,34.570425,9 Years and 10 Months,Yes,82.195693,47.86626156933074,Low_spent_Small_value_payments,295.0567958298957,2023-01-01
5212,CUS_0x5ed7,65558.36,5715.196667,9,8,18,9,"Auto Loan, Credit-Builder Loan, Home Equity Lo...",48,28,...,_,2239.44,36.243498,8 Years and 6 Months,Yes,268.991795,276.00533331915557,Low_spent_Medium_value_payments,306.52253864374563,2023-03-01
6068,CUS_0x6ad0,8027.195,560.932917,10,5,20,2,"Personal Loan, and Student Loan",30,28,...,Bad,1720.19,28.904384,10 Years and 0 Months,Yes,8.961406,32.29192211507269,Low_spent_Small_value_payments,304.8399634460265,2024-02-01
6345,CUS_0x6f0a,8461.285,699.107083,9,9,20,5,"Debt Consolidation Loan, Auto Loan, Home Equit...",27,28,...,Bad,3822.75,39.239666,5 Years and 7 Months,Yes,34.027858,84.72281313801153,!@9#%8,241.1600368347789,2023-05-01
8398,CUS_0x8d90,14826.93,916.763418,6,6,17,9,"Debt Consolidation Loan, Student Loan, Credit-...",19,28,...,Bad,3732.22,27.566375,7 Years and 10 Months,Yes,172.667290,63.39586805598342,!@9#%8,224.60867409670243,2024-07-01
8602,CUS_0x90a2,16578.29,1423.524167,8,10,21,8,"Personal Loan, Auto Loan, Payday Loan, Not Spe...",48,28,...,Bad,3566.19,22.272513,12 Years and 4 Months,Yes,77.759406,59.5589548191002,Low_spent_Small_value_payments,295.0340555593212,2024-11-01
10067,CUS_0xa672,51113.85000000001,4356.487500,7,9,30,9,"Auto Loan, Credit-Builder Loan, Home Equity Lo...",33,28,...,_,3164.78,36.625704,0 Years and 8 Months,Yes,214.843877,107.71251870647973,Low_spent_Large_value_payments,383.09235386443436,2023-11-01
11216,CUS_0xb756,17625.61,1406.800833,9,10,18,7,"Debt Consolidation Loan, Not Specified, Not Sp...",38,28,...,Bad,4491.22,33.096367,3 Years and 6 Months,Yes,87.873529,77.1364503793182,Low_spent_Large_value_payments,245.6701042285109,2023-12-01


In [141]:
financials_df['Changed_Credit_Limit'] = financials_df['Changed_Credit_Limit'].replace('_', np.nan)

In [144]:
financials_df['Changed_Credit_Limit'].astype(float).describe()

count    12246.000000
mean        10.398582
std          6.799253
min         -6.490000
25%          5.370000
50%          9.410000
75%         14.940000
max         36.970000
Name: Changed_Credit_Limit, dtype: float64

In [147]:
financials_df[financials_df['Changed_Credit_Limit'].astype(float)<0]['Changed_Credit_Limit']

194                    -1.64
270                    -2.16
331                    -3.97
346                     -3.5
356                    -4.75
                ...         
12290                  -0.56
12310                   -3.5
12331                  -1.19
12454    -0.4900000000000002
12487                  -5.37
Name: Changed_Credit_Limit, Length: 204, dtype: object

In [154]:
len(financials_df[financials_df['Changed_Credit_Limit'].astype(str).str.len() > 5])

372

In [158]:
financials_df[financials_df['Changed_Credit_Limit'].astype(str).str.len() > 5]['Changed_Credit_Limit']

35        3.3699999999999988
61         4.050000000000002
94        0.9000000000000004
149      0.13999999999999968
230        5.130000000000001
                ...         
12403      5.949999999999998
12416     1.3200000000000005
12432     5.2799999999999985
12454    -0.4900000000000002
12488     2.2200000000000006
Name: Changed_Credit_Limit, Length: 372, dtype: object

In [162]:
financials_df['Credit_Mix'].value_counts(dropna=False)

Credit_Mix
Standard    4497
Good        3032
_           2611
Bad         2360
Name: count, dtype: int64

In [167]:
financials_df['Outstanding_Debt'].value_counts(dropna=False)

Outstanding_Debt
1360.45    3
460.46     3
1151.7     3
1109.03    3
101.29     2
          ..
39.28      1
258.33     1
4266.37    1
1582.34    1
455.55     1
Name: count, Length: 12213, dtype: int64

In [173]:
pattern = r'^\d+\.\d{1,2}$'
financials_df[~financials_df['Outstanding_Debt'].astype(str).str.match(pattern)]['Outstanding_Debt']

50       2699.17_
78        642.42_
98        755.17_
281        865.3_
430       149.92_
           ...   
12216     289.59_
12328     108.67_
12367    1478.07_
12392     345.22_
12478    3783.35_
Name: Outstanding_Debt, Length: 139, dtype: object

In [178]:
financials_df[~financials_df['Outstanding_Debt'].astype(str).str.endswith('_')]['Outstanding_Debt'].astype(float).describe()

count    12361.000000
mean      1427.075792
std       1155.358597
min          0.230000
25%        566.210000
50%       1165.770000
75%       1949.730000
max       4998.070000
Name: Outstanding_Debt, dtype: float64

In [180]:
financials_df['Credit_Utilization_Ratio'].describe()

count    12500.000000
mean        32.349265
std          5.156815
min         20.100770
25%         28.066517
50%         32.418953
75%         36.623650
max         48.199824
Name: Credit_Utilization_Ratio, dtype: float64

In [182]:
financials_df['Credit_History_Age'].value_counts()

Credit_History_Age
16 Years and 2 Months     79
19 Years and 5 Months     77
16 Years and 0 Months     76
19 Years and 9 Months     76
18 Years and 10 Months    74
                          ..
3 Years and 7 Months       1
2 Years and 7 Months       1
27 Years and 7 Months      1
12 Years and 1 Months      1
28 Years and 1 Months      1
Name: count, Length: 393, dtype: int64

In [185]:
financials_df['Payment_of_Min_Amount'].value_counts(dropna=False)

Payment_of_Min_Amount
Yes    6571
No     4491
NM     1438
Name: count, dtype: int64

In [186]:
financials_df['Total_EMI_per_month'].describe()

count    12500.000000
mean      1488.394291
std       8561.449910
min          0.000000
25%         31.496968
50%         72.887628
75%        169.634826
max      81971.000000
Name: Total_EMI_per_month, dtype: float64

In [189]:
financials_df['Amount_invested_monthly'].value_counts(dropna=False)

Amount_invested_monthly
__10000__             558
0.0                    23
142.77246834677234      1
216.3477247078263       1
119.17239358493993      1
                     ... 
520.9152764430545       1
55.70892053568165       1
263.02911730661117      1
70.09134433396301       1
207.28433299472718      1
Name: count, Length: 11921, dtype: int64

In [192]:
filtered_df = financials_df[financials_df['Amount_invested_monthly'].astype(str)!='__10000__']

In [196]:
filtered_df[~filtered_df['Amount_invested_monthly'].astype(str).str.replace('.', '', 1).str.isnumeric()]['Amount_invested_monthly']

Series([], Name: Amount_invested_monthly, dtype: object)

In [198]:
filtered_df['Amount_invested_monthly'].astype(float).describe()

count    11942.000000
mean       194.614399
std        200.625580
min          0.000000
25%         71.358455
50%        127.409240
75%        234.608156
max       1977.326102
Name: Amount_invested_monthly, dtype: float64

In [200]:
financials_df['Payment_Behaviour'].value_counts(dropna=False)

Payment_Behaviour
Low_spent_Small_value_payments      3202
High_spent_Medium_value_payments    2242
Low_spent_Medium_value_payments     1686
High_spent_Large_value_payments     1683
High_spent_Small_value_payments     1389
Low_spent_Large_value_payments      1300
!@9#%8                               998
Name: count, dtype: int64

In [203]:
financials_df[~financials_df['Monthly_Balance'].astype(str).str.replace('.', '', 1).str.isnumeric()]['Monthly_Balance']

2003    __-333333333333333333333333333__
Name: Monthly_Balance, dtype: object

In [206]:
financials_df[financials_df['Monthly_Balance']!='__-333333333333333333333333333__']['Monthly_Balance'].astype(float).describe()

count    12499.000000
mean       403.920915
std        213.926191
min          0.382558
25%        270.144230
50%        339.749646
75%        473.376147
max       1463.792328
Name: Monthly_Balance, dtype: float64

In [209]:
lms_loan_daily_df.columns.to_list()

['loan_id',
 'Customer_ID',
 'loan_start_date',
 'tenure',
 'installment_num',
 'loan_amt',
 'due_amt',
 'paid_amt',
 'overdue_amt',
 'balance',
 'snapshot_date']

In [220]:
lms_loan_daily_df.describe(include='all')

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
count,137500,137500,137500,137500.0,137500.000000,137500.0,137500.000000,137500.000000,137500.000000,137500.000000,137500
unique,12500,12500,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35
top,CUS_0xffd_2024_03_01,CUS_0xffd,2024-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01
freq,11,11,5973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5539
mean,NaN,NaN,NaN,10.0,5.000000,10000.0,909.090909,711.810909,871.905455,5871.905455,NaN
std,NaN,NaN,NaN,0.0,3.162289,0.0,287.480833,485.726859,2002.672544,3070.777575,NaN
min,NaN,NaN,NaN,10.0,0.000000,10000.0,0.000000,0.000000,0.000000,0.000000,NaN
25%,NaN,NaN,NaN,10.0,2.000000,10000.0,1000.000000,0.000000,0.000000,3000.000000,NaN
50%,NaN,NaN,NaN,10.0,5.000000,10000.0,1000.000000,1000.000000,0.000000,7000.000000,NaN
75%,NaN,NaN,NaN,10.0,8.000000,10000.0,1000.000000,1000.000000,0.000000,8000.000000,NaN


In [222]:
lms_loan_daily_df

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date
0,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,0,10000,0.0,0.0,0.0,10000.0,2023-05-01
1,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,1,10000,1000.0,1000.0,0.0,9000.0,2023-06-01
2,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,2,10000,1000.0,1000.0,0.0,8000.0,2023-07-01
3,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,3,10000,1000.0,0.0,1000.0,8000.0,2023-08-01
4,CUS_0x1000_2023_05_01,CUS_0x1000,2023-05-01,10,4,10000,1000.0,2000.0,0.0,6000.0,2023-09-01
...,...,...,...,...,...,...,...,...,...,...,...
137495,CUS_0xffd_2024_03_01,CUS_0xffd,2024-03-01,10,6,10000,1000.0,1000.0,0.0,4000.0,2024-09-01
137496,CUS_0xffd_2024_03_01,CUS_0xffd,2024-03-01,10,7,10000,1000.0,1000.0,0.0,3000.0,2024-10-01
137497,CUS_0xffd_2024_03_01,CUS_0xffd,2024-03-01,10,8,10000,1000.0,1000.0,0.0,2000.0,2024-11-01
137498,CUS_0xffd_2024_03_01,CUS_0xffd,2024-03-01,10,9,10000,1000.0,1000.0,0.0,1000.0,2024-12-01


In [226]:
# Construct expected loan_id
lms_loan_daily_df["expected_loan_id"] = (
    lms_loan_daily_df["Customer_ID"] + "_" +
    lms_loan_daily_df["loan_start_date"].astype(str).str.replace("-", "_")
)

# Check
lms_loan_daily_df[
    lms_loan_daily_df["loan_id"] != lms_loan_daily_df["expected_loan_id"]
].copy()

,loan_id,Customer_ID,loan_start_date,tenure,installment_num,loan_amt,due_amt,paid_amt,overdue_amt,balance,snapshot_date,expected_loan_id


loan_id is a simple concat of Customer_ID and loan_start_date

In [291]:
attributes_df['Age'].value_counts()

Age
26      364
35      361
32      360
39      360
38      345
       ... 
1066      1
6516      1
4710      1
1087      1
7805      1
Name: count, Length: 301, dtype: int64

In [315]:
financials_df['Outstanding_Debt'].value_counts()

Outstanding_Debt
1360.45    3
460.46     3
1151.7     3
1109.03    3
101.29     2
          ..
39.28      1
258.33     1
4266.37    1
1582.34    1
455.55     1
Name: count, Length: 12213, dtype: int64

In [316]:
financials_df["Outstanding_Debt"].describe()

count       12500
unique      12213
top       1360.45
freq            3
Name: Outstanding_Debt, dtype: object

In [269]:
financials_df[financials_df["Num_Credit_Inquiries"] % 1 != 0]

,Customer_ID,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,snapshot_date


In [266]:
financials_df.columns.to_list()

['Customer_ID',
 'Annual_Income',
 'Monthly_Inhand_Salary',
 'Num_Bank_Accounts',
 'Num_Credit_Card',
 'Interest_Rate',
 'Num_of_Loan',
 'Type_of_Loan',
 'Delay_from_due_date',
 'Num_of_Delayed_Payment',
 'Changed_Credit_Limit',
 'Num_Credit_Inquiries',
 'Credit_Mix',
 'Outstanding_Debt',
 'Credit_Utilization_Ratio',
 'Credit_History_Age',
 'Payment_of_Min_Amount',
 'Total_EMI_per_month',
 'Amount_invested_monthly',
 'Payment_Behaviour',
 'Monthly_Balance',
 'snapshot_date']

## Silver v0

In [317]:
# def process_silver_tables(bronze_dir: str, silver_dir: str, spark: SparkSession):
#     """
#     Process Bronze Parquet tables into Silver tables (cleaned & standardized).

#     Args:
#         bronze_dir (str): Base path of Bronze tables.
#         silver_dir (str): Base path to save Silver tables.
#         spark (SparkSession): Active Spark session.
#     """

#     # === Silver processing rules per dataset ===
#     datasets = {
#         "features_attributes": {
#             "path": f"{bronze_dir}/features_attributes",
#             "types": {
#                 "Customer_ID": StringType(),
#                 "Name": StringType(),
#                 "Age": IntegerType(),
#                 "SSN": StringType(),
#                 "Occupation": StringType(),
#                 "snapshot_date": DateType()
#             },
#             "drop_nulls": ["Customer_ID"],  # enforce key existence
#         },
#         "features_financials": {
#             "path": f"{bronze_dir}/features_financials",
#             "types": {
#                 "Customer_ID": StringType(),
#                 "Annual_Income": FloatType(),
#                 "Monthly_Inhand_Salary": FloatType(),
#                 "Num_Bank_Accounts": IntegerType(),
#                 "Num_Credit_Card": IntegerType(),
#                 "Interest_Rate": IntegerType(),
#                 "Num_of_Loan": IntegerType(),
#                 "Type_of_Loan": StringType(),
#                 "Delay_from_due_date": IntegerType(),
#                 "Num_of_Delayed_Payment": IntegerType(),
#                 "Changed_Credit_Limit": FloatType(),
#                 "Num_Credit_Inquiries": IntegerType(),
#                 "Credit_Mix": StringType(),
#                 "Outstanding_Debt": FloatType(),
#                 "Credit_Utilization_Ratio": FloatType(),
#                 "Credit_History_Age": StringType(),  # parsed further below
#                 "Payment_of_Min_Amount": StringType(),
#                 "Total_EMI_per_month": FloatType(),
#                 "Amount_invested_monthly": FloatType(),
#                 "Payment_Behaviour": StringType(),
#                 "Monthly_Balance": FloatType(),
#                 "snapshot_date": DateType()
#             },
#             "drop_nulls": ["Customer_ID"]
#         },
#         "feature_clickstream": {
#             "path": f"{bronze_dir}/feature_clickstream",
#             "types": {
#                 **{f"fe_{i}": FloatType() for i in range(1, 21)},
#                 "Customer_ID": StringType(),
#                 "snapshot_date": DateType()
#             },
#             "drop_nulls": ["Customer_ID"]
#         },
#         "lms_loan_daily": {
#             "path": f"{bronze_dir}/lms_loan_daily",
#             "types": {
#                 "loan_id": StringType(),
#                 "Customer_ID": StringType(),
#                 "loan_start_date": DateType(),
#                 "tenure": IntegerType(),
#                 "installment_num": IntegerType(),
#                 "loan_amt": FloatType(),
#                 "due_amt": FloatType(),
#                 "paid_amt": FloatType(),
#                 "overdue_amt": FloatType(),
#                 "balance": FloatType(),
#                 "snapshot_date": DateType()
#             },
#             "drop_nulls": ["Customer_ID"]
#         }
#     }

#     # === Process each dataset ===
#     for table_name, config in datasets.items():
#         print(f"Processing Silver table: {table_name}...")

#         # Load Bronze parquet
#         df = spark.read.parquet(config["path"])
#         #df = df.drop("snapshot_date_dup")  # keep only the original

#         # Drop rows missing critical keys
#         for key in config["drop_nulls"]:
#             df = df.filter(col(key).isNotNull())

#         # === Dataset-specific rules ===
#         if table_name == "features_attributes":
            
#             # Clean Age column: remove trailing underscores and handle negative values
#             df = df.withColumn(
#                 "Age_cleaned",
#                 regexp_replace(col("Age").cast("string"), "_", "")  # remove underscores
#             )
            
#             # Convert to integer and replace negative values with null
#             df = df.withColumn(
#                 "Age_cleaned",
#                 when(col("Age_cleaned").cast("int") < 0, None).otherwise(col("Age_cleaned").cast(IntegerType()))
#             )
            
#             # Create a flag for valid ages (0-120)
#             df = df.withColumn(
#                 "Age_valid",
#                 when((col("Age_cleaned") >= 0) & (col("Age_cleaned") <= 120), lit(1)).otherwise(lit(0))
#             )

#             # Replace Age with Age_cleaned
#             df = df.withColumn("Age", col("Age_cleaned")).drop("Age_cleaned")
            
#             # Clean SSN (keep only valid regex pattern)
#             df = df.withColumn(
#                 "SSN",
#                 when(col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$"), col("SSN")).otherwise(lit(None))
#             )

#             # Replace placeholder occupation
#             df = df.withColumn(
#                 "Occupation",
#                 when(col("Occupation") == "_______", lit(None)).otherwise(col("Occupation"))
#             )

#         if table_name == "features_financials":

#             # Remove any trailing underscores and convert to float
#             df = df.withColumn(
#                 "Annual_Income_cleaned",
#                 regexp_replace(col("Annual_Income"), "_$", "").cast(FloatType())
#             )
            
#             # Round to 2 decimal places
#             df = df.withColumn(
#                 "Annual_Income_cleaned",
#                 round(col("Annual_Income_cleaned"), 2)
#             )
            
#             # Replace original column
#             df = df.withColumn("Annual_Income", col("Annual_Income_cleaned")).drop("Annual_Income_cleaned")

#             # Clean Num_Bank_Accounts
#             df = df.withColumn(
#                 "Num_Bank_Accounts_cleaned",
#                 when(col("Num_Bank_Accounts") < 0, None)         # replace negative values with null
#                 .when(col("Num_Bank_Accounts") > 11, lit(11))   # cap at 11
#                 .otherwise(col("Num_Bank_Accounts"))            # leave other values as-is
#             )
            
#             # Replace original column and drop the temporary
#             df = df.withColumn("Num_Bank_Accounts", col("Num_Bank_Accounts_cleaned")) \
#                    .drop("Num_Bank_Accounts_cleaned")

#             # Cap large invalid Num_Credit_Card
#             df = df.withColumn(
#                 "Num_Credit_Card",
#                 when(col("Num_Credit_Card") > 11, lit(11)).otherwise(col("Num_Credit_Card"))
#             )

#             # Cap large invalid Interest_Rate
#             df = df.withColumn(
#                 "Interest_Rate",
#                 when(col("Interest_Rate") > 34, lit(34)).otherwise(col("Interest_Rate"))
#             )

#             # Remove trailing underscores, cast to integer, handle negatives and cap values
#             df = df.withColumn(
#                 "Num_of_Loan",
#                 when(
#                     regexp_replace(col("Num_of_Loan").cast("string"), "_", "") == "", None
#                 ).otherwise(
#                     regexp_replace(col("Num_of_Loan").cast("string"), "_", "").cast("int")
#                 )
#             )
            
#             # Replace negatives with null and cap values > 9
#             df = df.withColumn(
#                 "Num_of_Loan",
#                 when(col("Num_of_Loan") < 0, None)
#                 .when(col("Num_of_Loan") > 9, lit(9))
#                 .otherwise(col("Num_of_Loan"))
#             )           

#             # Remove trailing underscores, cast to integer and cap values
#             df = df.withColumn(
#                 "Num_of_Delayed_Payment",
#                 when(
#                     regexp_replace(col("Num_of_Delayed_Payment").cast("string"), "_", "") == "", None
#                 ).otherwise(
#                     regexp_replace(col("Num_of_Delayed_Payment").cast("string"), "_", "").cast("int")
#                 )
#             )
            
#             # Replace negatives with null and cap values > 9
#             df = df.withColumn(
#                 "Num_of_Delayed_Payment",
#                 when(col("Num_of_Delayed_Payment") > 28, lit(28))
#                 .otherwise(col("Num_of_Delayed_Payment"))
#             )   

#             # Clean Changed_Credit_Limit
#             df = df.withColumn(
#                 "Changed_Credit_Limit",
#                 when(col("Changed_Credit_Limit") == "_", lit(None))  # replace '_' with NaN
#                 .otherwise(col("Changed_Credit_Limit").cast("float"))  # cast valid numbers
#             )
            
#             df = df.withColumn(
#                 "Changed_Credit_Limit",
#                 round(col("Changed_Credit_Limit"), 2)  # enforce 2 decimal places
#             )

#             df = df.withColumn(
#                 "Credit_Mix",
#                 when(col("Credit_Mix") == "_", None).otherwise(col("Credit_Mix"))
#             )

#             df = df.withColumn(
#                 "Outstanding_Debt",
#                 regexp_replace(col("Outstanding_Debt").cast("string"), "_", "").cast("float")
#             )
            
#             # Extract years and months as integers
#             df = df.withColumn("Credit_History_Years", regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Years", 1).cast(IntegerType()))
#             df = df.withColumn("Credit_History_Months", regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Months", 1).cast(IntegerType()))
        
#             # Compute total months
#             df = df.withColumn(
#                 "Credit_History_Age_Months",
#                 (col("Credit_History_Years") * 12 + col("Credit_History_Months")).cast(IntegerType())
#             )

#             df = df.withColumn(
#                 "Payment_of_Min_Amount_Flag",
#                 when(col("Payment_of_Min_Amount") == "Yes", 1)
#                 .when(col("Payment_of_Min_Amount") == "No", 0)
#                 .otherwise(None).cast(IntegerType())
#             )

#             df = df.withColumn(
#                 "Amount_invested_monthly",
#                 regexp_replace(col("Amount_invested_monthly").cast("string"), "^_+|_+$", "").cast("float")
#             )

#             df = df.withColumn(
#                 "Payment_Behaviour",
#                 when(col("Payment_Behaviour") == "!@9#%8", None)  # replace only the junk value with null
#                 .otherwise(col("Payment_Behaviour"))  # keep all other values unchanged
#             )

#             df = df.withColumn(
#                 "Monthly_Balance",
#                 when(col("Monthly_Balance") == "__-333333333333333333333333333__", None)  # replace only the junk value with null
#                 .otherwise(col("Monthly_Balance"))  # keep all other values unchanged
#             )
        
#         # Enforce data types
#         for col_name, dtype in config["types"].items():
#             if col_name in df.columns:
#                 df = df.withColumn(col_name, col(col_name).cast(dtype))
            
#         # Write out to Silver
#         output_path = f"{silver_dir}/{table_name}"
#         (
#             df.write
#             .mode("overwrite")
#             .partitionBy("snapshot_date")
#             .parquet(output_path)
#         )

#         print(f"✅ Saved Silver table: {output_path}")


## Silver

In [342]:
# === Helper functions ===

def clean_numeric_column(df: DataFrame, col_name: str, min_val=None, max_val=None, remove_trailing_underscore=True) -> DataFrame:
    """
    Clean a numeric column: remove trailing underscores, cast to int/float, replace negatives with null, cap max.
    """
    temp_col = f"{col_name}_cleaned"
    
    # Remove trailing underscores if required
    if remove_trailing_underscore:
        df = df.withColumn(temp_col, regexp_replace(col(col_name).cast("string"), "_", ""))
    else:
        df = df.withColumn(temp_col, col(col_name).cast("string"))
    
    # Convert to numeric
    df = df.withColumn(
        temp_col,
        when(col(temp_col) == "", None).otherwise(col(temp_col).cast(FloatType()))
    )
    
    # Apply min/max caps
    if min_val is not None:
        df = df.withColumn(temp_col, when(col(temp_col) < min_val, None).otherwise(col(temp_col)))
    if max_val is not None:
        df = df.withColumn(temp_col, when(col(temp_col) > max_val, lit(max_val)).otherwise(col(temp_col)))
    
    # Replace original
    df = df.withColumn(col_name, col(temp_col)).drop(temp_col)
    return df

def clean_string_column(df: DataFrame, col_name: str, junk_values: list = None, placeholder=None) -> DataFrame:
    """
    Replace junk or placeholder values with None.
    """
    temp_col = col_name
    if junk_values:
        for junk in junk_values:
            df = df.withColumn(temp_col, when(col(temp_col) == junk, None).otherwise(col(temp_col)))
    if placeholder:
        df = df.withColumn(temp_col, when(col(temp_col) == placeholder, None).otherwise(col(temp_col)))
    return df

# === Dataset-specific cleaning ===

def clean_features_attributes(df: DataFrame) -> DataFrame:
    # Clean Age
    df = clean_numeric_column(df, "Age", min_val=0, max_val=120)
    df = df.withColumn("Age_valid", when((col("Age") >= 0) & (col("Age") <= 120), lit(1)).otherwise(lit(0)))
    
    # Clean SSN
    df = df.withColumn("SSN", when(col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$"), col("SSN")).otherwise(lit(None)))
    
    # Clean Occupation
    df = clean_string_column(df, "Occupation", placeholder="_______")
    
    return df

def clean_features_financials(df: DataFrame) -> DataFrame:
    # Clean Annual_Income
    df = clean_numeric_column(df, "Annual_Income")
    df = df.withColumn("Annual_Income", round(col("Annual_Income"), 2))
    
    # Num_Bank_Accounts
    df = clean_numeric_column(df, "Num_Bank_Accounts", min_val=0, max_val=11)
    
    # Num_Credit_Card, Interest_Rate
    df = clean_numeric_column(df, "Num_Credit_Card", max_val=11)
    df = clean_numeric_column(df, "Interest_Rate", max_val=34)

    # Create array of strings representation for Type_of_Loan, remove leading 'and'
    df = df.withColumn(
        "Type_of_Loan_list",
        F.when(
            F.col("Type_of_Loan").isNotNull(),
            F.split(
                F.regexp_replace(F.col("Type_of_Loan"), r"\band\s+", ""),
                r",\s*"
            )
        ).otherwise(F.array())  # empty array instead of null
    )

    # Derive correct "Num_of_Loan" values 
    df = df.withColumn("Num_of_Loan", coalesce(size("Type_of_Loan_list"), lit(0)))

    
    # Num_of_Loan
    #df = clean_numeric_column(df, "Num_of_Loan", min_val=0, max_val=9)
    
    # Num_of_Delayed_Payment
    df = clean_numeric_column(df, "Num_of_Delayed_Payment", max_val=28)
    
    # Changed_Credit_Limit
    df = df.withColumn("Changed_Credit_Limit",
                       when(col("Changed_Credit_Limit") == "_", None)
                       .otherwise(round(col("Changed_Credit_Limit").cast("float"), 2)))
    
    # Credit_Mix
    df = clean_string_column(df, "Credit_Mix", junk_values=["_"])
    
    # Outstanding_Debt
    df = clean_numeric_column(df, "Outstanding_Debt")
    
    # Credit_History_Age -> years, months
    df = df.withColumn("Credit_History_Years", regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Years", 1).cast(IntegerType()))
    df = df.withColumn("Credit_History_Months", regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Months", 1).cast(IntegerType()))
    df = df.withColumn("Credit_History_Age_Months", (col("Credit_History_Years") * 12 + col("Credit_History_Months")).cast(IntegerType()))
    
    # Payment_of_Min_Amount_Flag
    # df = df.withColumn("Payment_of_Min_Amount_Flag",
    #                    when(col("Payment_of_Min_Amount") == "Yes", 1)
    #                    .when(col("Payment_of_Min_Amount") == "No", 0)
    #                    .otherwise(None).cast(IntegerType()))
    
    # Amount_invested_monthly
    df = df.withColumn("Amount_invested_monthly",
                       regexp_replace(col("Amount_invested_monthly").cast("string"), "^_+|_+$", "").cast(FloatType()))
    
    # Payment_Behaviour
    df = clean_string_column(df, "Payment_Behaviour", junk_values=["!@9#%8"])
    
    # Monthly_Balance
    df = clean_string_column(df, "Monthly_Balance", junk_values=["__-333333333333333333333333333__"])
    
    return df

def no_cleaning(df):
    return df

# === Main processing function ===

def process_silver_tables(bronze_dir: str, silver_dir: str, spark: SparkSession):
    datasets = {
        "features_attributes": {
            "path": f"{bronze_dir}/features_attributes",
            "types": {
                "Customer_ID": StringType(),
                "Name": StringType(),
                "Age": IntegerType(),
                "SSN": StringType(),
                "Occupation": StringType(),
                "snapshot_date": DateType()
            },
            "drop_nulls": ["Customer_ID"],
            "clean_func": clean_features_attributes
        },
        "features_financials": {
            "path": f"{bronze_dir}/features_financials",
            "types": {
                "Customer_ID": StringType(),
                "Annual_Income": FloatType(),
                "Monthly_Inhand_Salary": FloatType(),
                "Num_Bank_Accounts": IntegerType(),
                "Num_Credit_Card": IntegerType(),
                "Interest_Rate": IntegerType(),
                "Num_of_Loan": IntegerType(),
                "Type_of_Loan": StringType(),
                "Delay_from_due_date": IntegerType(),
                "Num_of_Delayed_Payment": IntegerType(),
                "Changed_Credit_Limit": FloatType(),
                "Num_Credit_Inquiries": IntegerType(),
                "Credit_Mix": StringType(),
                "Outstanding_Debt": FloatType(),
                "Credit_Utilization_Ratio": FloatType(),
                "Credit_History_Age": StringType(),
                "Payment_of_Min_Amount": StringType(),
                "Total_EMI_per_month": FloatType(),
                "Amount_invested_monthly": FloatType(),
                "Payment_Behaviour": StringType(),
                "Monthly_Balance": FloatType(),
                "snapshot_date": DateType()
            },
            "drop_nulls": ["Customer_ID"],
            "clean_func": clean_features_financials
        },
        "feature_clickstream": {
            "path": f"{bronze_dir}/feature_clickstream",
            "types": {
                **{f"fe_{i}": FloatType() for i in range(1, 21)},
                "Customer_ID": StringType(),
                "snapshot_date": DateType()
            },
            "drop_nulls": ["Customer_ID"],
            "clean_func": no_cleaning
        },
        "lms_loan_daily": {
            "path": f"{bronze_dir}/lms_loan_daily",
            "types": {
                "loan_id": StringType(),
                "Customer_ID": StringType(),
                "loan_start_date": DateType(),
                "tenure": IntegerType(),
                "installment_num": IntegerType(),
                "loan_amt": FloatType(),
                "due_amt": FloatType(),
                "paid_amt": FloatType(),
                "overdue_amt": FloatType(),
                "balance": FloatType(),
                "snapshot_date": DateType()
            },
            "drop_nulls": ["Customer_ID"],
            "clean_func": no_cleaning 
        }
    }
    
    for table_name, config in datasets.items():
        print(f"Processing Silver table: {table_name}...")
        
        # Load Bronze
        df = spark.read.parquet(config["path"])
        
        # Drop rows missing keys
        for key in config["drop_nulls"]:
            df = df.filter(col(key).isNotNull())
        
        # Apply dataset-specific cleaning
        df = config["clean_func"](df)

        # Enforce types
        for col_name, dtype in config["types"].items():
            if col_name in df.columns:
                df = df.withColumn(col_name, col(col_name).cast(dtype))        
        
        # Write to Silver
        output_path = f"{silver_dir}/{table_name}"
        df.write.mode("overwrite").partitionBy("snapshot_date").parquet(output_path)
        print(f"✅ Saved Silver table: {output_path}")


## Silver v2 (parameterized with YAML)

In [4]:
import yaml
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, when, lit, regexp_replace, split, size, coalesce, array, regexp_extract, round
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

# === Helper functions ===

def clean_numeric_column(df: DataFrame, col_name: str, min_val=None, max_val=None, remove_trailing_underscore=True) -> DataFrame:
    temp_col = f"{col_name}_cleaned"
    if remove_trailing_underscore:
        df = df.withColumn(temp_col, regexp_replace(col(col_name).cast("string"), "_", ""))
    else:
        df = df.withColumn(temp_col, col(col_name).cast("string"))
    df = df.withColumn(temp_col, when(col(temp_col) == "", None).otherwise(col(temp_col).cast(FloatType())))
    if min_val is not None:
        df = df.withColumn(temp_col, when(col(temp_col) < min_val, None).otherwise(col(temp_col)))
    if max_val is not None:
        df = df.withColumn(temp_col, when(col(temp_col) > max_val, lit(max_val)).otherwise(col(temp_col)))
    df = df.withColumn(col_name, col(temp_col)).drop(temp_col)
    return df

def clean_string_column(df: DataFrame, col_name: str, junk_values: list = None, placeholder=None) -> DataFrame:
    temp_col = col_name
    if junk_values:
        for junk in junk_values:
            df = df.withColumn(temp_col, when(col(temp_col) == junk, None).otherwise(col(temp_col)))
    if placeholder:
        df = df.withColumn(temp_col, when(col(temp_col) == placeholder, None).otherwise(col(temp_col)))
    return df

# Dataset-specific cleaning functions (original working ones)
def clean_features_attributes(df: DataFrame) -> DataFrame:
    df = clean_numeric_column(df, "Age", min_val=0, max_val=120)
    df = df.withColumn("Age_valid", when((col("Age") >= 0) & (col("Age") <= 120), lit(1)).otherwise(lit(0)))
    df = df.withColumn("SSN", when(col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$"), col("SSN")).otherwise(lit(None)))
    df = clean_string_column(df, "Occupation", placeholder="_______")
    return df

def clean_features_financials(df: DataFrame) -> DataFrame:
    df = clean_numeric_column(df, "Annual_Income")
    df = df.withColumn("Annual_Income", round(col("Annual_Income"), 2))
    df = clean_numeric_column(df, "Num_Bank_Accounts", min_val=0, max_val=11)
    df = clean_numeric_column(df, "Num_Credit_Card", max_val=11)
    df = clean_numeric_column(df, "Interest_Rate", max_val=34)
    df = clean_numeric_column(df, "Num_of_Delayed_Payment", max_val=28)
    df = df.withColumn("Changed_Credit_Limit", when(col("Changed_Credit_Limit") == "_", None)
                       .otherwise(round(col("Changed_Credit_Limit").cast("float"), 2)))
    df = clean_string_column(df, "Credit_Mix", junk_values=["_"])
    df = clean_numeric_column(df, "Outstanding_Debt")
    df = df.withColumn("Credit_History_Years", regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Years", 1).cast(IntegerType()))
    df = df.withColumn("Credit_History_Months", regexp_extract(col("Credit_History_Age"), r"(\d+)\s+Months", 1).cast(IntegerType()))
    df = df.withColumn("Credit_History_Age_Months", (col("Credit_History_Years")*12 + col("Credit_History_Months")).cast(IntegerType()))
    df = df.withColumn(
        "Type_of_Loan_list",
        when(
            col("Type_of_Loan").isNotNull(),
            split(regexp_replace(col("Type_of_Loan"), r"\band\s+", ""), r",\s*")
        ).otherwise(array())
    )
    df = df.withColumn("Num_of_Loan", coalesce(size("Type_of_Loan_list"), lit(0)))
    df = df.withColumn("Amount_invested_monthly",
                       regexp_replace(col("Amount_invested_monthly").cast("string"), "^_+|_+$", "").cast(FloatType()))
    df = clean_string_column(df, "Payment_Behaviour", junk_values=["!@9#%8"])
    df = clean_string_column(df, "Monthly_Balance", junk_values=["__-333333333333333333333333333__"])
    return df

def no_cleaning(df: DataFrame) -> DataFrame:
    return df

# === Main processing function ===

def process_silver_tables(spark: SparkSession, config_path="/app/config/silver_config.yaml"):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    bronze_dir = config["directories"]["bronze_dir"]
    silver_dir = config["directories"]["silver_dir"]

    for table_name, dataset_config in config["datasets"].items():
        print(f"Processing Silver table: {table_name}...")

        # Load Bronze
        df = spark.read.parquet(f"{bronze_dir}/{dataset_config['path']}")

        # Drop rows missing keys
        for key in dataset_config.get("drop_nulls", []):
            df = df.filter(col(key).isNotNull())

        # Apply cleaning logic
        if table_name == "features_attributes":
            df = clean_features_attributes(df)
        elif table_name == "features_financials":
            df = clean_features_financials(df)
        elif table_name in ["feature_clickstream", "lms_loan_daily"]:
            df = no_cleaning(df)

        # Cast types
        type_map = {"string": StringType(), "int": IntegerType(), "float": FloatType(), "date": DateType()}
        for col_name, dtype_str in dataset_config.get("types", {}).items():
            if col_name in df.columns:
                df = df.withColumn(col_name, col(col_name).cast(type_map[dtype_str]))

        # Write to Silver
        output_path = f"{silver_dir}/{table_name}"
        df.write.mode("overwrite").partitionBy("snapshot_date").parquet(output_path)
        print(f"✅ Saved Silver table: {output_path}")


## EDA 3

In [83]:
loans  = spark.read.parquet(f"/app/datamart/silver/lms_loan_daily/*")

In [85]:
loans.show()

+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+
|             loan_id|Customer_ID|loan_start_date|tenure|installment_num|loan_amt|due_amt|paid_amt|overdue_amt|balance|
+--------------------+-----------+---------------+------+---------------+--------+-------+--------+-----------+-------+
|CUS_0x1009_2025_0...| CUS_0x1009|     2025-01-01|    10|              0| 10000.0|    0.0|     0.0|        0.0|10000.0|
|CUS_0x100b_2024_0...| CUS_0x100b|     2024-03-01|    10|             10| 10000.0| 1000.0|  1000.0|        0.0|    0.0|
|CUS_0x102e_2024_0...| CUS_0x102e|     2024-04-01|    10|              9| 10000.0| 1000.0|     0.0|     7000.0| 8000.0|
|CUS_0x1038_2024_1...| CUS_0x1038|     2024-10-01|    10|              3| 10000.0| 1000.0|  1000.0|        0.0| 7000.0|
|CUS_0x103e_2024_1...| CUS_0x103e|     2024-12-01|    10|              1| 10000.0| 1000.0|  1000.0|        0.0| 9000.0|
|CUS_0x104f_2024_1...| CUS_0x104f|     2

In [90]:
loans.filter(F.col("loan_start_date") > F.to_date(F.lit("2023-03-01"))).select("Customer_ID").distinct().count()

10963

In [91]:
12500-10963

1537

In [98]:
loans.groupBy("loan_id", "Customer_ID", "loan_start_date").agg(F.max((F.col("overdue_amt") > 0).cast("int")).alias("label_default")).show()

+--------------------+-----------+---------------+-------------+
|             loan_id|Customer_ID|loan_start_date|label_default|
+--------------------+-----------+---------------+-------------+
|CUS_0x122c_2024_0...| CUS_0x122c|     2024-08-01|            0|
|CUS_0x1595_2024_0...| CUS_0x1595|     2024-07-01|            0|
|CUS_0x16b1_2024_0...| CUS_0x16b1|     2024-06-01|            0|
|CUS_0x19f3_2024_0...| CUS_0x19f3|     2024-03-01|            0|
|CUS_0x1cb8_2024_0...| CUS_0x1cb8|     2024-05-01|            0|
|CUS_0x1f57_2024_1...| CUS_0x1f57|     2024-12-01|            0|
|CUS_0x2963_2024_1...| CUS_0x2963|     2024-12-01|            0|
|CUS_0x3195_2024_0...| CUS_0x3195|     2024-04-01|            0|
|CUS_0x32be_2024_0...| CUS_0x32be|     2024-07-01|            0|
|CUS_0x3c55_2025_0...| CUS_0x3c55|     2025-01-01|            0|
|CUS_0x4577_2024_0...| CUS_0x4577|     2024-05-01|            1|
|CUS_0x487a_2024_0...| CUS_0x487a|     2024-03-01|            1|
|CUS_0x5217_2024_1...| CU

## Gold v0

In [ ]:
from pyspark.sql import functions as F

def process_gold_tables(spark, base_path="datamart"):
    # --- Load Silver Tables ---
    attrs = spark.read.parquet(f"{base_path}/silver/features_attributes/*")
    fins  = spark.read.parquet(f"{base_path}/silver/features_financials/*")
    clicks = spark.read.parquet(f"{base_path}/silver/features_clickstream/*")
    loans  = spark.read.parquet(f"{base_path}/silver/lms_loans_daily/*")

    # --- Truncate early loans: only loans after 2023-03-01 ---
    loans = loans.filter(F.col("loan_start_date") > F.to_date(F.lit("2023-03-01")))

    # --- Label Store ---
    label_store = (
        loans.groupBy("loan_id", "Customer_ID", "loan_start_date")
             .agg(F.max((F.col("overdue_amt") > 0).cast("int")).alias("label_default"))
    )
    label_store.write.mode("overwrite").parquet(f"{base_path}/gold/label_store")

    # --- Feature Store (static features: attributes + financials) ---
    static_feats = (
        attrs.alias("a")
        .join(fins.alias("f"),
              on=["Customer_ID", "snapshot_date"],
              how="inner")
    )

    # --- Clickstream Aggregates (relative to loan_start_date) ---
    clicks = clicks.withColumn("snapshot_date", F.to_date("snapshot_date"))
    loans = loans.withColumn("loan_start_date", F.to_date("loan_start_date"))

    click_feats = (
        clicks.alias("c")
        .join(loans.select("Customer_ID", "loan_start_date").distinct().alias("l"),
              on="Customer_ID")
        .where(F.col("c.snapshot_date") < F.col("l.loan_start_date"))
    )

    # Compute months difference
    click_feats = click_feats.withColumn(
        "months_diff",
        F.months_between("l.loan_start_date", "c.snapshot_date")
    )

    # Aggregate L1M / L3M (example)
    agg_exprs = [
        F.avg("c.fe_1").alias("avg_fe1_L3M"),
        F.sum("c.fe_1").alias("sum_fe1_L3M"),
        # add other features as needed
    ]

    click_aggs = (
        click_feats.groupBy("Customer_ID", "loan_start_date")
                   .agg(*agg_exprs)
    )

    # --- Final Feature Store ---
    feature_store = (
        label_store.select("Customer_ID", "loan_start_date")  # anchor keys
        .join(static_feats, 
              (static_feats.Customer_ID == label_store.Customer_ID) &
              (static_feats.snapshot_date == label_store.loan_start_date),
              "left")
        .join(click_aggs,
              on=["Customer_ID", "loan_start_date"],
              how="left")
    )

    feature_store.write.mode("overwrite").parquet(f"{base_path}/gold/feature_store")

    print("✅ Gold tables created at", f"{base_path}/gold")


In [111]:
def aggregate_clickstream(click_feats, feature_cols):
    """
    Aggregate clickstream features relative to loan_start_date.
    - LM: exact snapshot value from loan_start_date - 1 month
    - L3M: average and sum over the last 3 months (exclusive of loan_start_date)
    """
    agg_exprs = []
    for col in feature_cols:
        # ---- L3M aggregates ----
        agg_exprs.append(F.avg(F.col(f"c.{col}")).alias(f"avg_{col}_L3M"))
        agg_exprs.append(F.sum(F.col(f"c.{col}")).alias(f"sum_{col}_L3M"))

        # ---- LM (exact last month value) ----
        agg_exprs.append(
            F.max(
                F.when(
                    F.col("c.snapshot_date") == F.add_months(F.col("l.loan_start_date"), -1),
                    F.col(f"c.{col}")
                )
            ).alias(f"{col}_LM")
        )

    click_aggs = (
        click_feats
        .where(F.col("c.snapshot_date") >= F.add_months(F.col("l.loan_start_date"), -3))  # restrict to L3M window
        .groupBy("Customer_ID", "loan_start_date")
        .agg(*agg_exprs)
    )

    return click_aggs


def process_gold_tables(silver_base_path, gold_base_path, spark):
    """
    Build gold tables (label_store + feature_store) from silver datasets.
    """
    # ----------------------------------------------------------------------
    # 1. Load silver datasets
    # ----------------------------------------------------------------------
    attrs = spark.read.parquet(f"{silver_base_path}/features_attributes")
    fins = spark.read.parquet(f"{silver_base_path}/features_financials")
    clicks = spark.read.parquet(f"{silver_base_path}/feature_clickstream")
    loans = spark.read.parquet(f"{silver_base_path}/lms_loan_daily")

    # ----------------------------------------------------------------------
    # 2. Type casting
    # ----------------------------------------------------------------------
    clicks = clicks.withColumn("snapshot_date", F.to_date("snapshot_date"))
    loans = loans.withColumn("loan_start_date", F.to_date("loan_start_date")) \
                 .withColumn("snapshot_date", F.to_date("snapshot_date"))

    # ----------------------------------------------------------------------
    # 3. Filter away early loans (truncate before 2023-03-01)
    # ----------------------------------------------------------------------
    loans = loans.filter(F.col("loan_start_date") > F.to_date(F.lit("2023-03-01")))

    # ----------------------------------------------------------------------
    # 4. Label store: default flag
    # ----------------------------------------------------------------------
    label_store = (
        loans.groupBy("Customer_ID", "loan_start_date")
             .agg(F.max(F.when(F.col("overdue_amt") > 0, 1).otherwise(0)).alias("default_flag"))
    )

    # ----------------------------------------------------------------------
    # 5. Static features (attributes + financials)
    # Align snapshot_date = loan_start_date
    # ----------------------------------------------------------------------
    static_feats = (
        attrs.alias("a")
             .join(fins.alias("f"), on=["Customer_ID", "snapshot_date"], how="inner")
    )

    # ----------------------------------------------------------------------
    # 6. Clickstream aggregation (LM / L3M)
    # ----------------------------------------------------------------------
    # Join clickstream with loans for each Customer_ID
    click_feats = (
        clicks.alias("c")
              .join(loans.select("Customer_ID", "loan_start_date").distinct().alias("l"),
                    on="Customer_ID")
              .where(F.col("c.snapshot_date") < F.col("l.loan_start_date"))
    )

    # Aggregate features: fe_1 ... fe_20
    feature_cols = [f"fe_{i}" for i in range(1, 21)]
    click_aggs = aggregate_clickstream(click_feats, feature_cols)

    # ----------------------------------------------------------------------
    # 7. Feature store = static + clickstream, aligned with label store loans
    # ----------------------------------------------------------------------
    feature_store = (
        label_store.select("Customer_ID", "loan_start_date")  # anchor keys
        .join(static_feats,
              (static_feats.Customer_ID == label_store.Customer_ID) &
              (static_feats.snapshot_date == label_store.loan_start_date),
              "left")
        .drop(static_feats.Customer_ID)
        .drop(static_feats.snapshot_date)
        .join(click_aggs, on=["Customer_ID", "loan_start_date"], how="left")
    )

    # ----------------------------------------------------------------------
    # 8. Save to gold tables
    # ----------------------------------------------------------------------
    label_store.write.mode("overwrite").parquet(f"{gold_base_path}/label_store")
    print(f"✅ Labels saved to {gold_base_path}/label_store")
    
    feature_store.write.mode("overwrite").parquet(f"{gold_base_path}/feature_store")
    print(f"✅ Features saved to {gold_base_path}/feature_store")

    return feature_store, label_store 


## Gold

In [270]:
def aggregate_clickstream(clicks, feature_cols, loans):
    """
    Robust clickstream aggregation.
    - clicks: raw clickstream dataframe (with columns: Customer_ID, snapshot_date, fe_1..fe_N)
    - loans: loans dataframe (Customer_ID, loan_start_date)
    Returns a DataFrame keyed by (Customer_ID, loan_start_date) with L3M and LM features.
    """

    c = clicks.alias("c")
    l = loans.select("Customer_ID", "loan_start_date").alias("l")

    # JOIN once (we'll filter per window below)
    joined = c.join(l, on="Customer_ID")

    # ---- L3M: avg & sum over last 3 months strictly before loan_start_date ----
    l3m = (
        joined
        .where((F.col("c.snapshot_date") >= F.add_months(F.col("l.loan_start_date"), -3)) &
               (F.col("c.snapshot_date") <  F.col("l.loan_start_date")))
        .groupBy(
            F.col("c.Customer_ID").alias("Customer_ID"),
            F.col("l.loan_start_date").alias("loan_start_date")
        )
        .agg(
            *[F.avg(F.col(f"c.{col}")).alias(f"avg_{col}_L3M") for col in feature_cols],
            *[F.sum(F.col(f"c.{col}")).alias(f"sum_{col}_L3M") for col in feature_cols]
        )
    )

    # ---- LM: exact value for snapshot_date == loan_start_date - 1 month
    # Use groupBy + first() to guard against multiple rows per month (just take first)
    lm = (
        joined
        .where(F.col("c.snapshot_date") == F.add_months(F.col("l.loan_start_date"), -1))
        .groupBy(
            F.col("c.Customer_ID").alias("Customer_ID"),
            F.col("l.loan_start_date").alias("loan_start_date")
        )
        .agg(
            *[F.first(F.col(f"c.{col}")).alias(f"{col}_LM") for col in feature_cols]
        )
    )

    # Join L3M and LM (one row per key)
    click_aggs = l3m.join(lm, on=["Customer_ID", "loan_start_date"], how="left")

    return click_aggs

def process_gold_tables(silver_base_path, gold_base_path, spark):
    """
    Build gold tables (label_store + feature_store) from silver datasets.
    Explicitly select only the desired columns from each silver table.
    """
    # ----------------------------------------------------------------------
    # 1. Load silver datasets
    # ----------------------------------------------------------------------
    attrs = spark.read.parquet(f"{silver_base_path}/features_attributes")
    fins = spark.read.parquet(f"{silver_base_path}/features_financials")
    clicks = spark.read.parquet(f"{silver_base_path}/feature_clickstream")
    loans = spark.read.parquet(f"{silver_base_path}/lms_loan_daily")

    # ----------------------------------------------------------------------
    # 2. Filter away early loans (truncate on or before 2023-03-01)
    # ----------------------------------------------------------------------
    loans = loans.filter(F.col("loan_start_date") > F.to_date(F.lit("2023-03-01")))

    # ----------------------------------------------------------------------
    # 3. Label store: binary default flag
    # ----------------------------------------------------------------------
    label_store = (
        loans.groupBy("Customer_ID", "loan_start_date")
             .agg(F.max(F.when(F.col("overdue_amt") > 0, 1).otherwise(0)).alias("default_flag"))
    )

    # ----------------------------------------------------------------------
    # 4. Static features (attributes + financials)
    # ----------------------------------------------------------------------
    # Select only desired columns (identifiers required for joining will be dropped later)
    static_feats = (
        attrs.alias("a")
             .join(fins.alias("f"), on=["Customer_ID", "snapshot_date"], how="inner")
             .select(
                 "Customer_ID",
                 "snapshot_date",
                 # Attributes
                 "Age", 
                 "Age_valid",
                 #"SSN", # Identifier not used for joins and not useful for ML
                 "Occupation", # OHE later
                 # Financials
                 "Annual_Income",
                 "Monthly_Inhand_Salary",
                 "Num_Bank_Accounts",
                 "Num_Credit_Card",
                 "Interest_Rate",
                 "Num_of_Loan",
                 "Type_of_Loan_list", #OHE later
                 "Delay_from_due_date",
                 "Num_of_Delayed_Payment",
                 "Changed_Credit_Limit",
                 "Num_Credit_Inquiries",
                 "Credit_Mix", # OHE later
                 "Outstanding_Debt",
                 "Credit_Utilization_Ratio",
                 "Credit_History_Age_Months",
                 "Payment_of_Min_Amount", # OHE later
                 "Amount_invested_monthly",
                 "Payment_Behaviour", # OHE later
                 "Monthly_Balance"
             )
    )

    # ----------------------------------------------------------------------
    # 5. Clickstream aggregation (LM / L3M)
    # ----------------------------------------------------------------------
    # click_feats = (
    #     clicks.alias("c")
    #           .join(loans.select("Customer_ID", "loan_start_date").distinct().alias("l"),
    #                 on="Customer_ID")
    #           .where(F.col("c.snapshot_date") < F.col("l.loan_start_date"))
    # )

    feature_cols = [f"fe_{i}" for i in range(1, 21)]
    click_aggs = aggregate_clickstream(clicks, feature_cols, loans)

    # ----------------------------------------------------------------------
    # 6. Feature store = static + clickstream, aligned with label store loans
    # ----------------------------------------------------------------------
    feature_store = (
        label_store.select("Customer_ID", "loan_start_date")  # anchor keys
        .join(static_feats,
              (static_feats.Customer_ID == label_store.Customer_ID) &
              (static_feats.snapshot_date == label_store.loan_start_date),
              "left")
        .drop(static_feats.Customer_ID)
        .drop(static_feats.snapshot_date)
        .join(click_aggs, on=["Customer_ID", "loan_start_date"], how="left")
    )
    
    # --- Handle missing clickstream features ---
    click_cols = [c for c in feature_store.columns if c.endswith("_L3M") or c.endswith("_LM")]
    feature_store = feature_store.fillna(0, subset=click_cols)    

    # ----------------------------------------------------------------------
    # --- ML specific handling ---
    # ----------------------------------------------------------------------

    # ----------------------------------------------------------------------
    # 7. Missing value handling
    # ----------------------------------------------------------------------
    median_age = feature_store.approxQuantile("Age", [0.5], 0.01)[0]
    median_bank_accounts = feature_store.approxQuantile("Num_Bank_Accounts", [0.5], 0.01)[0]
    
    feature_store = (
        feature_store
        # Numeric imputations
        .withColumn("Age", F.when(F.col("Age").isNull(), F.lit(median_age)).otherwise(F.col("Age")))
        .withColumn("Num_Bank_Accounts", 
                    F.when(F.col("Num_Bank_Accounts").isNull(), F.lit(median_bank_accounts))
                     .otherwise(F.col("Num_Bank_Accounts")))      
        .withColumn("Changed_Credit_Limit", 
                    F.when(F.col("Changed_Credit_Limit").isNull(), F.lit(0))
                     .otherwise(F.col("Changed_Credit_Limit")))
        .withColumn("Monthly_Balance", 
                    F.when(F.col("Monthly_Balance").isNull(), F.lit(0))
                     .otherwise(F.col("Monthly_Balance")))
        # Categorical imputations
        .withColumn("Occupation", F.when(F.col("Occupation").isNull(), F.lit("Unknown")).otherwise(F.col("Occupation")))
        .withColumn("Credit_Mix", F.when(F.col("Credit_Mix").isNull(), F.lit("Unknown")).otherwise(F.col("Credit_Mix")))
        .withColumn("Payment_Behaviour", F.when(F.col("Payment_Behaviour").isNull(), F.lit("Unknown")).otherwise(F.col("Payment_Behaviour")))
    )

    # ----------------------------------------------------------------------
    # 8. Multi-hot encoding for Type_of_Loan_list
    # ----------------------------------------------------------------------

    with open("/app/config/features.yaml", "r") as f:
        config = yaml.safe_load(f)

    expected_loans = config["loan_types"]

    # Replace null/empty with ["No_Loan"]
    feature_store = feature_store.withColumn(
        "Type_of_Loan_list",
        F.when(F.size(F.col("Type_of_Loan_list")) == 0, F.array(F.lit("No_Loan")))
         .when(F.col("Type_of_Loan_list").isNull(), F.array(F.lit("No_Loan")))
         .otherwise(F.col("Type_of_Loan_list"))
    )

    # Collect unique values for sanity check
    unique_vals = [row[0] for row in feature_store.select(F.explode("Type_of_Loan_list")).distinct().collect()]
    unexpected = set(unique_vals) - set(expected_loans) - {"No_Loan"}

    if unexpected:
        print(f"⚠️ Unexpected loan types found (ignored in encoding): {unexpected}")
    else:
        print(f"✅ No unexpected loan types found in data.")

    # Multi-hot encode
    for loan in expected_loans + ["No_Loan"]:
        feature_store = feature_store.withColumn(
            f"LoanType_{loan.replace(' ', '_')}",
            F.when(F.array_contains(F.col("Type_of_Loan_list"), loan), F.lit(1)).otherwise(F.lit(0))
        )

    # Drop original list
    feature_store = feature_store.drop("Type_of_Loan_list")

    # ----------------------------------------------------------------------
    # 9. One-hot encoding for other categorical variables
    # ----------------------------------------------------------------------
    cat_cols = ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]
 
    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
    encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_OHE") for c in cat_cols]
    
    pipeline = Pipeline(stages=indexers + encoders)
    fitted_pipeline = pipeline.fit(feature_store)
    feature_store = fitted_pipeline.transform(feature_store)
    
    # ----------------------------------------------------------------------
    # Expand OHE columns into readable flat columns
    # ----------------------------------------------------------------------
    for c in cat_cols:
        # Get labels from the fitted StringIndexerModel
        labels = fitted_pipeline.stages[cat_cols.index(c)].labels  # list of category names
    
        # Convert vector to array so we can access individual indices
        arr = vector_to_array(f"{c}_OHE")
        feature_store = feature_store.withColumn(f"{c}_arr", arr)
    
        # Create one column per category, named after the label
        for i, label in enumerate(labels):
            clean_label = label.replace(" ", "_").replace("/", "_").replace("-", "_")
            feature_store = feature_store.withColumn(f"{c}_{clean_label}", F.col(f"{c}_arr")[i])
    
        # Drop intermediate columns
        feature_store = feature_store.drop(f"{c}_arr", f"{c}_OHE", f"{c}_idx", c)    
    
    # ----------------------------------------------------------------------
    # 10. Final cleanup
    # ----------------------------------------------------------------------
    # Drop identifiers not used for ML
    feature_store = feature_store.drop("Customer_ID", "loan_start_date")  
    label_store = label_store.drop("Customer_ID", "loan_start_date")
    
    # ----------------------------------------------------------------------
    # 11. Save to gold tables
    # ----------------------------------------------------------------------
    label_store.write.mode("overwrite").parquet(f"{gold_base_path}/label_store")
    print(f"✅ Labels saved to {gold_base_path}/label_store")
    
    feature_store.write.mode("overwrite").parquet(f"{gold_base_path}/feature_store")
    print(f"✅ Features saved to {gold_base_path}/feature_store")

    return feature_store, label_store


## Gold v2 (with splits)

In [296]:
def aggregate_clickstream(clicks, feature_cols, loans):
    """
    Robust clickstream aggregation.
    - clicks: raw clickstream dataframe (with columns: Customer_ID, snapshot_date, fe_1..fe_N)
    - loans: loans dataframe (Customer_ID, loan_start_date)
    Returns a DataFrame keyed by (Customer_ID, loan_start_date) with L3M and LM features.
    """

    c = clicks.alias("c")
    l = loans.select("Customer_ID", "loan_start_date").alias("l")

    # JOIN once (we'll filter per window below)
    joined = c.join(l, on="Customer_ID")

    # ---- L3M: avg & sum over last 3 months strictly before loan_start_date ----
    l3m = (
        joined
        .where((F.col("c.snapshot_date") >= F.add_months(F.col("l.loan_start_date"), -3)) &
               (F.col("c.snapshot_date") <  F.col("l.loan_start_date")))
        .groupBy(
            F.col("c.Customer_ID").alias("Customer_ID"),
            F.col("l.loan_start_date").alias("loan_start_date")
        )
        .agg(
            *[F.avg(F.col(f"c.{col}")).alias(f"avg_{col}_L3M") for col in feature_cols],
            *[F.sum(F.col(f"c.{col}")).alias(f"sum_{col}_L3M") for col in feature_cols]
        )
    )

    # ---- LM: exact value for snapshot_date == loan_start_date - 1 month
    # Use groupBy + first() to guard against multiple rows per month (just take first)
    lm = (
        joined
        .where(F.col("c.snapshot_date") == F.add_months(F.col("l.loan_start_date"), -1))
        .groupBy(
            F.col("c.Customer_ID").alias("Customer_ID"),
            F.col("l.loan_start_date").alias("loan_start_date")
        )
        .agg(
            *[F.first(F.col(f"c.{col}")).alias(f"{col}_LM") for col in feature_cols]
        )
    )

    # Join L3M and LM (one row per key)
    click_aggs = l3m.join(lm, on=["Customer_ID", "loan_start_date"], how="left")

    return click_aggs

def process_gold_tables(silver_base_path, gold_base_path, spark):
    """
    Build gold tables (label_store + feature_store) from silver datasets.
    Explicitly select only the desired columns from each silver table.
    """
    # ----------------------------------------------------------------------
    # 1. Load silver datasets
    # ----------------------------------------------------------------------
    attrs = spark.read.parquet(f"{silver_base_path}/features_attributes")
    fins = spark.read.parquet(f"{silver_base_path}/features_financials")
    clicks = spark.read.parquet(f"{silver_base_path}/feature_clickstream")
    loans = spark.read.parquet(f"{silver_base_path}/lms_loan_daily")

    # ----------------------------------------------------------------------
    # 2. Filter away early loans (truncate on or before 2023-03-01)
    # ----------------------------------------------------------------------
    loans = loans.filter(F.col("loan_start_date") > F.to_date(F.lit("2023-03-01")))

    # ----------------------------------------------------------------------
    # 3. Label store: binary default flag
    # ----------------------------------------------------------------------
    label_store = (
        loans.groupBy("Customer_ID", "loan_start_date")
             .agg(F.max(F.when(F.col("overdue_amt") > 0, 1).otherwise(0)).alias("default_flag"))
    )

    # ----------------------------------------------------------------------
    # 4. Static features (attributes + financials)
    # ----------------------------------------------------------------------
    # Select only desired columns (identifiers required for joining will be dropped later)
    static_feats = (
        attrs.alias("a")
             .join(fins.alias("f"), on=["Customer_ID", "snapshot_date"], how="inner")
             .select(
                 "Customer_ID",
                 "snapshot_date",
                 # Attributes
                 "Age", 
                 "Age_valid",
                 #"SSN", # Identifier not used for joins and not useful for ML
                 "Occupation", # OHE later
                 # Financials
                 "Annual_Income",
                 "Monthly_Inhand_Salary",
                 "Num_Bank_Accounts",
                 "Num_Credit_Card",
                 "Interest_Rate",
                 "Num_of_Loan",
                 "Type_of_Loan_list", #OHE later
                 "Delay_from_due_date",
                 "Num_of_Delayed_Payment",
                 "Changed_Credit_Limit",
                 "Num_Credit_Inquiries",
                 "Credit_Mix", # OHE later
                 "Outstanding_Debt",
                 "Credit_Utilization_Ratio",
                 "Credit_History_Age_Months",
                 "Payment_of_Min_Amount", # OHE later
                 "Amount_invested_monthly",
                 "Payment_Behaviour", # OHE later
                 "Monthly_Balance"
             )
    )

    # ----------------------------------------------------------------------
    # 5. Clickstream aggregation (LM / L3M)
    # ----------------------------------------------------------------------

    feature_cols = [f"fe_{i}" for i in range(1, 21)]
    click_aggs = aggregate_clickstream(clicks, feature_cols, loans)

    # ----------------------------------------------------------------------
    # 6. Feature store = static + clickstream, aligned with label store loans
    # ----------------------------------------------------------------------
    feature_store = (
        label_store.select("Customer_ID", "loan_start_date")  # anchor keys
        .join(static_feats,
              (static_feats.Customer_ID == label_store.Customer_ID) &
              (static_feats.snapshot_date == label_store.loan_start_date),
              "left")
        .drop(static_feats.Customer_ID)
        .drop(static_feats.snapshot_date)
        .join(click_aggs, on=["Customer_ID", "loan_start_date"], how="left")
    )
    
    # --- Handle missing clickstream features ---
    click_cols = [c for c in feature_store.columns if c.endswith("_L3M") or c.endswith("_LM")]
    feature_store = feature_store.fillna(0, subset=click_cols)    

    # ----------------------------------------------------------------------
    # 7. Define date-based splits
    # ----------------------------------------------------------------------
    max_date = feature_store.agg(F.max("loan_start_date")).first()[0]
    oot_start = max_date
    test_end = F.add_months(F.lit(oot_start), -1)
    test_start = F.add_months(test_end, -2)
    val_end = F.add_months(test_start, -1)
    val_start = F.add_months(val_end, -2)
    train_end = F.add_months(val_start, -1)

    # Split datasets
    train_df = feature_store.filter(F.col("loan_start_date") <= train_end)
    val_df = feature_store.filter((F.col("loan_start_date") >= val_start) & (F.col("loan_start_date") <= val_end))
    test_df = feature_store.filter((F.col("loan_start_date") >= test_start) & (F.col("loan_start_date") <= test_end))
    oot_df = feature_store.filter(F.col("loan_start_date") == oot_start)

    # Same splits for labels
    train_labels = label_store.join(train_df.select("Customer_ID", "loan_start_date"), on=["Customer_ID", "loan_start_date"])
    val_labels = label_store.join(val_df.select("Customer_ID", "loan_start_date"), on=["Customer_ID", "loan_start_date"])
    test_labels = label_store.join(test_df.select("Customer_ID", "loan_start_date"), on=["Customer_ID", "loan_start_date"])
    oot_labels = label_store.join(oot_df.select("Customer_ID", "loan_start_date"), on=["Customer_ID", "loan_start_date"])
    
    # ----------------------------------------------------------------------
    # --- ML specific handling ---
    # ----------------------------------------------------------------------

    # ----------------------------------------------------------------------
    # 7. Missing value handling
    # ----------------------------------------------------------------------
    median_age = feature_store.approxQuantile("Age", [0.5], 0.01)[0]
    median_bank_accounts = feature_store.approxQuantile("Num_Bank_Accounts", [0.5], 0.01)[0]
    
    feature_store = (
        feature_store
        # Numeric imputations
        .withColumn("Age", F.when(F.col("Age").isNull(), F.lit(median_age)).otherwise(F.col("Age")))
        .withColumn("Num_Bank_Accounts", 
                    F.when(F.col("Num_Bank_Accounts").isNull(), F.lit(median_bank_accounts))
                     .otherwise(F.col("Num_Bank_Accounts")))      
        .withColumn("Changed_Credit_Limit", 
                    F.when(F.col("Changed_Credit_Limit").isNull(), F.lit(0))
                     .otherwise(F.col("Changed_Credit_Limit")))
        .withColumn("Monthly_Balance", 
                    F.when(F.col("Monthly_Balance").isNull(), F.lit(0))
                     .otherwise(F.col("Monthly_Balance")))
        # Categorical imputations
        .withColumn("Occupation", F.when(F.col("Occupation").isNull(), F.lit("Unknown")).otherwise(F.col("Occupation")))
        .withColumn("Credit_Mix", F.when(F.col("Credit_Mix").isNull(), F.lit("Unknown")).otherwise(F.col("Credit_Mix")))
        .withColumn("Payment_Behaviour", F.when(F.col("Payment_Behaviour").isNull(), F.lit("Unknown")).otherwise(F.col("Payment_Behaviour")))
    )

    # ----------------------------------------------------------------------
    # 8. Missing value handling (fit on train)
    # ----------------------------------------------------------------------
    median_age = train_df.approxQuantile("Age", [0.5], 0.01)[0]
    median_bank_accounts = train_df.approxQuantile("Num_Bank_Accounts", [0.5], 0.01)[0]

    def impute_missing(df):
        return (
            df.withColumn("Age", F.when(F.col("Age").isNull(), F.lit(median_age)).otherwise(F.col("Age")))
              .withColumn("Num_Bank_Accounts", F.when(F.col("Num_Bank_Accounts").isNull(), F.lit(median_bank_accounts)).otherwise(F.col("Num_Bank_Accounts")))
              .withColumn("Changed_Credit_Limit", F.when(F.col("Changed_Credit_Limit").isNull(), F.lit(0)).otherwise(F.col("Changed_Credit_Limit")))
              .withColumn("Monthly_Balance", F.when(F.col("Monthly_Balance").isNull(), F.lit(0)).otherwise(F.col("Monthly_Balance")))
              .withColumn("Occupation", F.when(F.col("Occupation").isNull(), F.lit("Unknown")).otherwise(F.col("Occupation")))
              .withColumn("Credit_Mix", F.when(F.col("Credit_Mix").isNull(), F.lit("Unknown")).otherwise(F.col("Credit_Mix")))
              .withColumn("Payment_Behaviour", F.when(F.col("Payment_Behaviour").isNull(), F.lit("Unknown")).otherwise(F.col("Payment_Behaviour")))
        )

    train_df = impute_missing(train_df)
    val_df = impute_missing(val_df)
    test_df = impute_missing(test_df)
    oot_df = impute_missing(oot_df)

    # ----------------------------------------------------------------------
    # 9. Multi-hot encoding for Type_of_Loan_list
    # ----------------------------------------------------------------------
    with open("/app/config/features.yaml", "r") as f:
        config = yaml.safe_load(f)
    expected_loans = config["loan_types"]

    def encode_loans(df):
        df = df.withColumn(
            "Type_of_Loan_list",
            F.when(F.size(F.col("Type_of_Loan_list")) == 0, F.array(F.lit("No_Loan")))
             .when(F.col("Type_of_Loan_list").isNull(), F.array(F.lit("No_Loan")))
             .otherwise(F.col("Type_of_Loan_list"))
        )
    
        # --- sanity check ---
        unique_vals = [row[0] for row in df.select(F.explode("Type_of_Loan_list")).distinct().collect()]
        unexpected = set(unique_vals) - set(expected_loans) - {"No_Loan"}
        if unexpected:
            print(f"⚠️ Unexpected loan types found (ignored in encoding): {unexpected}")
        else:
            print(f"✅ No unexpected loan types found in data.")
    
        # --- multi-hot encoding ---
        for loan in expected_loans + ["No_Loan"]:
            df = df.withColumn(f"LoanType_{loan.replace(' ', '_')}",
                               F.when(F.array_contains(F.col("Type_of_Loan_list"), loan), 1).otherwise(0))
        df = df.drop("Type_of_Loan_list")
        return df

    train_df = encode_loans(train_df)
    val_df = encode_loans(val_df)
    test_df = encode_loans(test_df)
    oot_df = encode_loans(oot_df)

    # ----------------------------------------------------------------------
    # 10. One-hot encoding for other categorical variables (fit on train only)
    # ----------------------------------------------------------------------
    cat_cols = ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]
    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
    encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_OHE") for c in cat_cols]

    pipeline = Pipeline(stages=indexers + encoders)
    fitted_pipeline = pipeline.fit(train_df)

    train_df = fitted_pipeline.transform(train_df)
    val_df = fitted_pipeline.transform(val_df)
    test_df = fitted_pipeline.transform(test_df)
    oot_df = fitted_pipeline.transform(oot_df)

    # Expand OHE columns into readable flat columns
    def flatten_ohe(df):
        for c in cat_cols:
            labels = fitted_pipeline.stages[cat_cols.index(c)].labels
            arr = vector_to_array(f"{c}_OHE")
            df = df.withColumn(f"{c}_arr", arr)
            for i, label in enumerate(labels):
                clean_label = label.replace(" ", "_").replace("/", "_").replace("-", "_")
                df = df.withColumn(f"{c}_{clean_label}", F.col(f"{c}_arr")[i])
            df = df.drop(f"{c}_arr", f"{c}_OHE", f"{c}_idx", c)
        return df

    train_df = flatten_ohe(train_df)
    val_df = flatten_ohe(val_df)
    test_df = flatten_ohe(test_df)
    oot_df = flatten_ohe(oot_df)

    # ----------------------------------------------------------------------
    # 11. Drop identifiers
    # ----------------------------------------------------------------------
    train_df = train_df.drop("Customer_ID", "loan_start_date")
    val_df = val_df.drop("Customer_ID", "loan_start_date")
    test_df = test_df.drop("Customer_ID", "loan_start_date")
    oot_df = oot_df.drop("Customer_ID", "loan_start_date")

    train_labels = train_labels.drop("Customer_ID", "loan_start_date")
    val_labels = val_labels.drop("Customer_ID", "loan_start_date")
    test_labels = test_labels.drop("Customer_ID", "loan_start_date")
    oot_labels = oot_labels.drop("Customer_ID", "loan_start_date")

    # ----------------------------------------------------------------------
    # 12. Save splits
    # ----------------------------------------------------------------------
    splits = {
        "train": (train_df, train_labels),
        "val": (val_df, val_labels),
        "test": (test_df, test_labels),
        "OOT": (oot_df, oot_labels)
    }

    for name, (feat, lab) in splits.items():
        print(f"📊 {name.upper()} split: features shape=({feat.count()}, {len(feat.columns)}), labels shape=({lab.count()}, {len(lab.columns)})")
        
        feat.write.mode("overwrite").parquet(f"{gold_base_path}/feature_store/{name}")
        lab.write.mode("overwrite").parquet(f"{gold_base_path}/label_store/{name}")
        print(f"✅ {name} split saved: features -> {gold_base_path}/feature_store/{name}, labels -> {gold_base_path}/label_store/{name}")

    return splits

## Gold v3 (parameterized with yaml)

In [5]:
def aggregate_clickstream(clicks, feature_cols, loans):
    """
    Aggregates clickstream data (L3M avg/sum and LM values).
    """

    c = clicks.alias("c")
    l = loans.select("Customer_ID", "loan_start_date").alias("l")

    joined = c.join(l, on="Customer_ID")

    # ---- L3M: avg & sum over last 3 months strictly before loan_start_date ----
    l3m = (
        joined
        .where((F.col("c.snapshot_date") >= F.add_months(F.col("l.loan_start_date"), -3)) &
               (F.col("c.snapshot_date") <  F.col("l.loan_start_date")))
        .groupBy(
            F.col("c.Customer_ID").alias("Customer_ID"),
            F.col("l.loan_start_date").alias("loan_start_date")
        )
        .agg(
            *[F.avg(F.col(f"c.{col}")).alias(f"avg_{col}_L3M") for col in feature_cols],
            *[F.sum(F.col(f"c.{col}")).alias(f"sum_{col}_L3M") for col in feature_cols]
        )
    )

    # ---- LM: exact value for snapshot_date == loan_start_date - 1 month
    # Use groupBy + first() to guard against multiple rows per month (just take first)
    lm = (
        joined
        .where(F.col("c.snapshot_date") == F.add_months(F.col("l.loan_start_date"), -1))
        .groupBy(
            F.col("c.Customer_ID").alias("Customer_ID"),
            F.col("l.loan_start_date").alias("loan_start_date")
        )
        .agg(
            *[F.first(F.col(f"c.{col}")).alias(f"{col}_LM") for col in feature_cols]
        )
    )

    # Join L3M and LM (one row per key)
    click_aggs = l3m.join(lm, on=["Customer_ID", "loan_start_date"], how="left")

    return click_aggs

def process_gold_tables(spark, config_path="/app/config/gold_config.yaml"):
    """
    Build gold tables using YAML-driven configuration.
    Important note: Identifiers "Customer_ID" and "loan_start_date" are retained in
    feature and label stores to faciitate mapping of output to prediction. They will
    be dropped as part of the ML pipeline during training.
    """
    # ----------------------------------------------------------------------
    # 1. Load YAML configuration
    # ----------------------------------------------------------------------
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    silver_base_path = config["paths"]["silver_base"]
    gold_base_path = config["paths"]["gold_base"]
    splits_cfg = config["splits"]
    static_cfg = config["static_features"]
    click_cols = config["clickstream_features"]
    loan_cfg = config["loan_types"]
    cat_cols = config["categorical_features"]
    impute_cfg = config["imputation"]
    
    # ----------------------------------------------------------------------
    # 2. Load silver datasets
    # ----------------------------------------------------------------------
    attrs = spark.read.parquet(f"{silver_base_path}/features_attributes")
    fins = spark.read.parquet(f"{silver_base_path}/features_financials")
    clicks = spark.read.parquet(f"{silver_base_path}/feature_clickstream")
    loans = spark.read.parquet(f"{silver_base_path}/lms_loan_daily")

    # ----------------------------------------------------------------------
    # 3. Filter away early loans (truncate on or before 2023-03-01)
    # ----------------------------------------------------------------------
    loans = loans.filter(F.col("loan_start_date") > F.to_date(F.lit(splits_cfg["min_date"])))

    # ----------------------------------------------------------------------
    # 4. Label store: binary default flag
    # ----------------------------------------------------------------------
    label_store = (
        loans.groupBy("Customer_ID", "loan_start_date")
             .agg(F.max(F.when(F.col("overdue_amt") > 0, 1).otherwise(0)).alias("default_flag"))
    )

    # ----------------------------------------------------------------------
    # 5. Static features (attributes + financials)
    # ----------------------------------------------------------------------
    attr_cols = static_cfg["attributes"]
    fin_cols = static_cfg["financials"]

    static_feats = (
        attrs.alias("a")
             .join(fins.alias("f"), on=["Customer_ID", "snapshot_date"], how="inner")
             .select("Customer_ID", "snapshot_date", *attr_cols, *fin_cols)
    )

    # ----------------------------------------------------------------------
    # 6. Clickstream aggregation (LM / L3M)
    # ----------------------------------------------------------------------

    click_aggs = aggregate_clickstream(clicks, click_cols, loans)

    # ----------------------------------------------------------------------
    # 7. Feature store = static + clickstream, aligned with label store loans
    # ----------------------------------------------------------------------
    feature_store = (
        label_store.select("Customer_ID", "loan_start_date")  # anchor keys
        .join(static_feats,
              (static_feats.Customer_ID == label_store.Customer_ID) &
              (static_feats.snapshot_date == label_store.loan_start_date),
              "left")
        .drop(static_feats.Customer_ID)
        .drop(static_feats.snapshot_date)
        .join(click_aggs, on=["Customer_ID", "loan_start_date"], how="left")
    )
    
    # --- Handle missing clickstream features ---
    click_cols = [c for c in feature_store.columns if c.endswith("_L3M") or c.endswith("_LM")]
    feature_store = feature_store.fillna(0, subset=click_cols)    

    # ----------------------------------------------------------------------
    # 8. Define date-based splits
    # ----------------------------------------------------------------------
    max_date = feature_store.agg(F.max("loan_start_date")).first()[0]
    oot_months = splits_cfg["oot_months"]
    test_months = splits_cfg["test_months"]
    val_months = splits_cfg["val_months"]
    train_months = splits_cfg["train_months"]

    oot_start = max_date
    test_end = F.add_months(F.lit(oot_start), -oot_months)
    test_start = F.add_months(test_end, -test_months + 1)
    val_end = F.add_months(test_start, -1)
    val_start = F.add_months(val_end, -val_months + 1)
    train_start = F.add_months(val_start, -train_months + 1)

    train_df = feature_store.filter((F.col("loan_start_date") >= train_start) & (F.col("loan_start_date") <= val_start))
    val_df = feature_store.filter((F.col("loan_start_date") >= val_start) & (F.col("loan_start_date") <= val_end))
    test_df = feature_store.filter((F.col("loan_start_date") >= test_start) & (F.col("loan_start_date") <= test_end))
    oot_df = feature_store.filter(F.col("loan_start_date") == oot_start)

    # ----------------------------------------------------------------------
    # 9. Label alignment - defensive measure since label and feature store already aligned from above
    # ----------------------------------------------------------------------
    def align_labels(df):
        return label_store.join(df.select("Customer_ID", "loan_start_date"), on=["Customer_ID", "loan_start_date"])

    train_labels = align_labels(train_df)
    val_labels = align_labels(val_df)
    test_labels = align_labels(test_df)
    oot_labels = align_labels(oot_df)
    
    # ----------------------------------------------------------------------
    # --- ML specific handling ---
    # ----------------------------------------------------------------------

    # ----------------------------------------------------------------------
    # 10. Missing value handling
    # ----------------------------------------------------------------------
    num_impute = impute_cfg["numeric"]
    cat_impute = impute_cfg["categorical"]

    # Compute median values from training data
    median_vals = {}
    for col, method in num_impute.items():
        if method == "median":
            val = train_df.approxQuantile(col, [0.5], 0.01)
            median_vals[col] = val[0] if val else None
        else:
            median_vals[col] = method  # numeric constant

    def impute_missing(df):
        for col, val in median_vals.items():
            df = df.withColumn(col, F.when(F.col(col).isNull(), F.lit(val)).otherwise(F.col(col)))
        for col, val in cat_impute.items():
            df = df.withColumn(col, F.when(F.col(col).isNull(), F.lit(val)).otherwise(F.col(col)))
        return df

    train_df = impute_missing(train_df)
    val_df = impute_missing(val_df)
    test_df = impute_missing(test_df)
    oot_df = impute_missing(oot_df)

    # ----------------------------------------------------------------------
    # 11. Multi-hot encoding for Type_of_Loan_list
    # ----------------------------------------------------------------------
    expected_loans = loan_cfg["expected"]
    default_loan = loan_cfg["default"]

    def encode_loans(df):
        df = df.withColumn(
            "Type_of_Loan_list",
            F.when(F.size(F.col("Type_of_Loan_list")) == 0, F.array(F.lit(default_loan)))
             .when(F.col("Type_of_Loan_list").isNull(), F.array(F.lit(default_loan)))
             .otherwise(F.col("Type_of_Loan_list"))
        )
        unique_vals = [row[0] for row in df.select(F.explode("Type_of_Loan_list")).distinct().collect()]
        unexpected = set(unique_vals) - set(expected_loans) - {default_loan}
        if unexpected:
            print(f"⚠️ Unexpected loan types found: {unexpected}")
        else:
            print(f"✅ No unexpected loan types found in data.")

        for loan in expected_loans + [default_loan]:
            df = df.withColumn(f"LoanType_{loan.replace(' ', '_')}",
                               F.when(F.array_contains(F.col("Type_of_Loan_list"), loan), 1).otherwise(0))
        return df.drop("Type_of_Loan_list")

    train_df = encode_loans(train_df)
    val_df = encode_loans(val_df)
    test_df = encode_loans(test_df)
    oot_df = encode_loans(oot_df)

    # ----------------------------------------------------------------------
    # 12. One-hot encoding for other categorical variables (fit on train only)
    # ----------------------------------------------------------------------
    indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
    encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_OHE") for c in cat_cols]

    pipeline = Pipeline(stages=indexers + encoders)
    fitted_pipeline = pipeline.fit(train_df)

    def apply_ohe(df):
        df = fitted_pipeline.transform(df)
        for c in cat_cols:
            labels = fitted_pipeline.stages[cat_cols.index(c)].labels
            arr = vector_to_array(f"{c}_OHE")
            df = df.withColumn(f"{c}_arr", arr)
            for i, label in enumerate(labels):
                clean_label = label.replace(" ", "_").replace("/", "_").replace("-", "_")
                df = df.withColumn(f"{c}_{clean_label}", F.col(f"{c}_arr")[i])
            df = df.drop(f"{c}_arr", f"{c}_OHE", f"{c}_idx", c)
        return df

    train_df = apply_ohe(train_df)
    val_df = apply_ohe(val_df)
    test_df = apply_ohe(test_df)
    oot_df = apply_ohe(oot_df)

    # ----------------------------------------------------------------------
    # 13. Drop identifiers --- Identifiers will only be dropped during training
    # ----------------------------------------------------------------------
    # for df in [train_df, val_df, test_df, oot_df]:
    #     df.drop("Customer_ID", "loan_start_date")

    # train_labels = train_labels.drop("Customer_ID", "loan_start_date")
    # val_labels = val_labels.drop("Customer_ID", "loan_start_date")
    # test_labels = test_labels.drop("Customer_ID", "loan_start_date")
    # oot_labels = oot_labels.drop("Customer_ID", "loan_start_date")

    # ----------------------------------------------------------------------
    # 14. Save splits
    # ----------------------------------------------------------------------
    splits = {
        "train": (train_df, train_labels),
        "val": (val_df, val_labels),
        "test": (test_df, test_labels),
        "OOT": (oot_df, oot_labels),
    }

    for name, (feat, lab) in splits.items():
        print(f"📊 {name.upper()} split: features shape=({feat.count()}, {len(feat.columns)}), labels shape=({lab.count()}, {len(lab.columns)})")
        feat.write.mode("overwrite").parquet(f"{gold_base_path}/feature_store/{name}")
        lab.write.mode("overwrite").parquet(f"{gold_base_path}/label_store/{name}")
        print(f"✅ {name} split saved.")

    return splits

## ML Consumption Test

In [278]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# ----------------------------------------------------------------------
# 1️⃣ Convert Spark DataFrames to pandas
# ----------------------------------------------------------------------
X_pd = X.toPandas()
y_pd = y.toPandas()

# Make sure your target column is named 'label'
if "label" not in y_pd.columns:
    y_pd = y_pd.rename(columns={y_pd.columns[0]: "label"})

# Drop any non-feature identifiers (if still present)
if "Customer_ID" in X_pd.columns:
    X_pd = X_pd.drop(columns=["Customer_ID"])

# ----------------------------------------------------------------------
# 2️⃣ Train-test split
# ----------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_pd, y_pd["label"], test_size=0.2, random_state=42
)

# ----------------------------------------------------------------------
# 3️⃣ Fit Logistic Regression model
# ----------------------------------------------------------------------
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# ----------------------------------------------------------------------
# 4️⃣ Evaluate
# ----------------------------------------------------------------------
y_train_pred = clf.predict(X_train)
y_train_pred_proba = clf.predict_proba(X_train)[:, 1]

print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Train AUC:", roc_auc_score(y_train, y_train_pred_proba))

y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Train Accuracy: 0.7127708095781072
Train AUC: 0.5300305742268215
Accuracy: 0.7140902872777017
AUC: 0.4922424486852799

Classification Report:
               precision    recall  f1-score   support

           0       0.71      1.00      0.83      1566
           1       0.00      0.00      0.00       627

    accuracy                           0.71      2193
   macro avg       0.36      0.50      0.42      2193
weighted avg       0.51      0.71      0.59      2193



/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre